# Constellation detection: classical milestone
Runnable classical pipeline and diagnostics based only on supplied images and references. This is a measured implementation milestone, not completion of every experiment in PLAN.md. The learned/HPC track remains separate.

## 1. Setup and input
Select `smoke`, `evaluate`, or `submission`. Supply a Drive directory or upload a dataset ZIP. An enclosing `participant/` folder is supported. No GPU is required.

In [ ]:
import sys, subprocess, os
if os.environ.get('CONSTELLATION_SKIP_INSTALL') != '1':
    subprocess.check_call([sys.executable,'-m','pip','install','numpy>=2.0','scipy>=1.14','opencv-python-headless>=4.10','Pillow>=10'])

In [ ]:
from pathlib import Path
import tempfile, zipfile, json
MODE = os.environ.get('CONSTELLATION_MODE','evaluate')
DATA = Path(os.environ.get('CONSTELLATION_DATA','/content/participant'))
if not DATA.exists():
    from google.colab import files
    uploads=files.upload()
    archive=next(Path(n) for n in uploads if n.endswith('.zip'))
    DATA=Path('/content/constellation_data');DATA.mkdir(exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        for member in z.infolist():
            if not (DATA/member.filename).resolve().is_relative_to(DATA.resolve()): raise ValueError('Unsafe ZIP path')
        z.extractall(DATA)
if not (DATA/'patterns').exists() and (DATA/'participant').exists(): DATA=DATA/'participant'
assert (DATA/'patterns').exists(), 'Set DATA to the folder containing patterns/train/validation'
DATA=DATA.resolve()
print('Data:',DATA,'Mode:',MODE)

In [ ]:
WORK=Path(tempfile.mkdtemp(prefix='constellation-notebook-'))
SOURCE={}
sys.path.insert(0,str(WORK))
OUTPUT=Path(os.environ.get('CONSTELLATION_OUTPUT',str(DATA/'outputs/notebook')));OUTPUT.mkdir(parents=True,exist_ok=True)

In [ ]:
# Embedded implementation: constellation/__init__.py
SOURCE['constellation/__init__.py'] = r'''"""Scene-independent constellation detection."""
'''
path=WORK/'constellation/__init__.py'
path.parent.mkdir(parents=True,exist_ok=True)
path.write_text(SOURCE['constellation/__init__.py'])

In [ ]:
# Embedded implementation: constellation/contracts.py
SOURCE['constellation/contracts.py'] = r'''from dataclasses import dataclass, field
import ast
import csv
import numpy as np

@dataclass
class ScenePrediction:
    patches: list
    constellation: str = 'unknown'
    diagnostics: dict = field(default_factory=dict)

def parse_cell(value):
    if str(value).strip() == '-1':
        return None
    p = ast.literal_eval(value)
    if len(p) != 3 or p[2] not in (0, 1) or not np.isfinite(p).all():
        raise ValueError(f'Invalid patch: {value}')
    return tuple(p)

def read_truth(path):
    with open(path, newline='') as f:
        rows = list(csv.DictReader(f))
    return {r['Id']: ScenePrediction([parse_cell(r[f'patch_{i:02}']) for i in range(1, int(r['n_patches'])+1)], r['constellation']) for r in rows}

def reward(distance):
    return np.clip((36 - np.asarray(distance)) / 24, 0, 1)

def evaluate(predictions, ground_truth):
    results = {}
    for name, truth in ground_truth.items():
        pred = predictions[name]
        if len(pred.patches) != len(truth.patches):
            raise ValueError('Query count mismatch')
        y = np.array([p is not None for p in truth.patches])
        z = np.array([p is not None for p in pred.patches])
        f1 = []
        for cls in (False, True):
            tp = np.sum((y == cls) & (z == cls))
            den = np.sum(y == cls) + np.sum(z == cls)
            f1.append(2*tp/den if den else 1.0)
        loc = [float(reward(np.linalg.norm(np.array(t[:2])-p[:2]))) if p is not None else 0.0 for p,t in zip(pred.patches, truth.patches) if t is not None]
        figure = np.array([t[:2] for t in truth.patches if t is not None and t[2] == 1]).reshape(-1,2)
        points = np.array([p[:2] for p in pred.patches if p is not None]).reshape(-1,2)
        total = 0.
        if len(figure) and len(points):
            d = np.linalg.norm(figure[:,None,:]-points[None,:,:], axis=2)
            used_f, used_p = set(), set()
            for flat in np.argsort(d, axis=None, kind='stable'):
                i,j = np.unravel_index(flat, d.shape)
                if i not in used_f and j not in used_p:
                    total += float(reward(d[i,j]))
                    used_f.add(i); used_p.add(j)
        m = dict(presence=float(np.mean(f1)), localization=float(np.mean(loc)) if loc else 1., recovery=total/len(figure) if len(figure) else 1., identification=float(pred.constellation==truth.constellation))
        m['score'] = sum(m[k]*w for k,w in [('presence',.25),('localization',.2),('recovery',.25),('identification',.3)])
        results[name] = m
    return {'scenes':results, 'mean':{k:float(np.mean([r[k] for r in results.values()])) for k in next(iter(results.values()))}, 'worst_score':min(r['score'] for r in results.values()), 'conventions':'Unofficial: empty class F1 and empty localization/recovery = 1; greedy distance ties use stable row-major order.'}

def write_submission(predictions, sample_submission, output_path):
    with open(sample_submission, newline='') as f:
        reader = csv.DictReader(f); fields = reader.fieldnames; rows = list(reader)
    if set(predictions) != {r['Id'] for r in rows}:
        raise ValueError('Scene set mismatch')
    for row in rows:
        p = predictions[row['Id']]
        if len(p.patches) != int(row['n_patches']):
            raise ValueError('Query count mismatch')
        for col in fields:
            if col.startswith('patch_'):
                idx = int(col.split('_')[1])-1
                point = p.patches[idx] if idx < len(p.patches) else None
                row[col] = '-1' if point is None else str((round(float(point[0]),3),round(float(point[1]),3),int(point[2])))
        row['constellation'] = p.constellation
    with open(output_path,'w',newline='') as f:
        writer = csv.DictWriter(f,fieldnames=fields); writer.writeheader(); writer.writerows(rows)
'''
path=WORK/'constellation/contracts.py'
path.parent.mkdir(parents=True,exist_ok=True)
path.write_text(SOURCE['constellation/contracts.py'])

In [ ]:
# Embedded implementation: constellation/dense.py
SOURCE['constellation/dense.py'] = r'''"""Slower exhaustive coarse appearance fallback; no source detector dependency."""
import cv2
import numpy as np

def dense_candidates(image,patch,per_pose=3):
    # Downsample only scene search. Candidate poses are reverified at full resolution.
    small=cv2.resize(image.astype(np.float32),None,fx=.5,fy=.5,interpolation=cv2.INTER_AREA)
    small=small-cv2.GaussianBlur(small,(0,0),2.)
    q=patch.astype(np.float32)
    q=q-cv2.GaussianBlur(q,(0,0),4.)
    coords=np.arange(-7,8,dtype=np.float32)*2
    xx,yy=np.meshgrid(coords,coords)
    candidates=[]
    for scale in (.75,.87,1.,1.15,1.33):
        # Largest inscribed square valid under every rotation at this scale.
        half=int(np.floor(15*scale/np.sqrt(2)/2))
        yy,xx=np.mgrid[-half:half+1,-half:half+1].astype(np.float32)*2
        for angle in np.arange(0,360,15):
            a=np.deg2rad(angle)
            mx=(15.5+(np.cos(a)*xx-np.sin(a)*yy)/scale).astype(np.float32)
            my=(15.5+(np.sin(a)*xx+np.cos(a)*yy)/scale).astype(np.float32)
            template=cv2.remap(q,mx,my,cv2.INTER_LINEAR)
            scores=cv2.matchTemplate(small,template,cv2.TM_CCOEFF_NORMED)
            for _ in range(per_pose):
                _,v,_,(x,y)=cv2.minMaxLoc(scores)
                candidates.append(((x+half)*2+.5,(y+half)*2+.5))
                scores[max(0,y-8):y+9,max(0,x-8):x+9]=-1
    return np.unique(np.array(candidates,dtype=np.float32),axis=0)
'''
path=WORK/'constellation/dense.py'
path.parent.mkdir(parents=True,exist_ok=True)
path.write_text(SOURCE['constellation/dense.py'])

In [ ]:
# Embedded implementation: constellation/finalize.py
SOURCE['constellation/finalize.py'] = r'''import cv2
import numpy as np
from .geometry import recognize
from .joint import recognize_joint
from .contracts import ScenePrediction

# Coarse-stage appearance scores are on a different scale from ECC-refined ones.
COARSE_CUTOFF = .65


def auxiliary_map(image):
    raw = image.astype(np.float32)
    dog = cv2.GaussianBlur(raw, (0, 0), 1) - cv2.GaussianBlur(raw, (0, 0), 8)
    response = cv2.dilate(dog, np.ones((25, 25), np.uint8))
    reference = np.sort(response.ravel()[::10])
    return (np.searchsorted(reference, response) / len(reference)).astype(np.float32)


def finalize(image, queries, raw_queries, patterns, threshold=.72, seed=6643):
    """Frozen milestone stage: one point per query, quad+triangle affine hashing."""
    aux = auxiliary_map(image); competing = []
    for stage, qs, cutoff in [('refined', queries, threshold),
                              ('coarse', raw_queries, COARSE_CUTOFF)]:
        ids = [i for i, q in enumerate(qs) if q[0][2] >= cutoff]
        name, m, d = recognize([qs[i][0][:2] for i in ids], patterns, seed=seed,
                               use_quads=True, shear_penalty=2., tolerance=18.,
                               auxiliary_map=aux)
        best = d['hypotheses'][0] if d.get('hypotheses') else {'score': -1e9, 'support': 0}
        competing.append((best['score'], name, d, stage))
    competing.sort(key=lambda x: (-x[0], x[1], x[3]))
    _, name, geometry, stage = competing[0]
    nodes = (np.array(geometry['hypotheses'][0]['nodes'])
             if geometry.get('hypotheses') and 'nodes' in geometry['hypotheses'][0]
             else np.empty((0, 2)))
    patches = []
    for q in queries:
        x, y, score, *_ = q[0]
        member = int(len(nodes) > 0 and np.linalg.norm(nodes - [x, y], axis=1).min() < 18)
        patches.append((x, y, member) if score >= threshold else None)
    return ScenePrediction(patches, name, {
        'geometry': geometry, 'selected_geometry_stage': stage,
        'competing_geometry': [{'stage': s, 'name': n, 'score': float(v)}
                               for v, n, d, s in competing]})


def finalize_joint(image, queries, raw_queries, patterns, threshold=.72, seed=6643,
                   tolerance=18., gap=.03, top_k=8, cap=80000, quad_share=0.,
                   member_radius=18., aux_weight=3., snap=True, snap_min_gap=0.):
    """Joint localization and recognition.

    Differs from `finalize` in three measured ways. Verification runs against the
    alternatives of appearance-ambiguous queries rather than one point each, and
    the winning fit's chosen alternative replaces the reported coordinate. Seeding
    uses triangle invariants only, because four-point quad invariants compound the
    ~5px template-to-sky model error. The hypothesis budget is raised, which only
    matters once that model error is represented.

    On 192 synthetic scenes at an unseen seed, identification is 0.474 against
    0.125 for `finalize`, and relocation produced 230 fixes with 0 regressions.
    On the three labelled scenes the weighted mean is 0.729 against 0.676.
    """
    aux = auxiliary_map(image)
    competing = []
    for stage, qs, cutoff in [('refined', queries, threshold),
                              ('coarse', raw_queries, COARSE_CUTOFF)]:
        ids = [i for i, q in enumerate(qs) if len(q) and q[0][2] >= cutoff]
        if len(ids) < 3:
            continue
        name, chosen, d = recognize_joint(
            [qs[i] for i in ids], patterns, seed=seed, tolerance=tolerance, gap=gap,
            top_k=top_k, cap=cap, quad_share=quad_share, aux_weight=aux_weight,
            models=('affine',), shear_penalty=2., auxiliary_map=aux)
        best = d['hypotheses'][0] if d.get('hypotheses') else {'score': -1e9}
        if not snap or (d.get('score_gap') or 0.) < snap_min_gap:
            chosen = {}
        competing.append((best.get('score', -1e9), name,
                          {ids[k]: v for k, v in chosen.items()}, d, stage))
    if not competing:
        return ScenePrediction([None] * len(queries), 'unknown',
                               {'reason': 'too few candidate points'})
    competing.sort(key=lambda x: (-x[0], x[1], x[4]))
    _, name, chosen, geometry, stage = competing[0]
    nodes = (np.array(geometry['hypotheses'][0].get('nodes', [])).reshape(-1, 2)
             if geometry.get('hypotheses') else np.empty((0, 2)))
    patches = []
    for i, q in enumerate(queries):
        if not len(q):
            patches.append(None); continue
        x, y, score = float(q[0][0]), float(q[0][1]), float(q[0][2])
        if i in chosen:
            x, y = chosen[i]
        member = int(len(nodes) > 0
                     and np.linalg.norm(nodes - [x, y], axis=1).min() < member_radius)
        patches.append((x, y, member) if score >= threshold else None)
    return ScenePrediction(patches, name, {
        'geometry': geometry, 'selected_geometry_stage': stage,
        'relocated_queries': sorted(chosen),
        'competing_geometry': [{'stage': s, 'name': n, 'score': float(v)}
                               for v, n, _, _, s in competing]})
'''
path=WORK/'constellation/finalize.py'
path.parent.mkdir(parents=True,exist_ok=True)
path.write_text(SOURCE['constellation/finalize.py'])

In [ ]:
# Embedded implementation: constellation/geometry.py
SOURCE['constellation/geometry.py'] = r'''"""Triangle-hash hypotheses with independent, one-to-one affine verification."""
import hashlib
from itertools import combinations
import numpy as np
from scipy.spatial import cKDTree
from scipy.stats import binom
from .quad import quad_jobs

def consolidate(points,radius=3.):
    # Anchor grouping avoids transitive chains. Radius is deliberately far below
    # scoring tolerance; close sources separated by >3 pixels stay distinct.
    unique=[];groups=[]
    for p in points:
        distances=np.linalg.norm(np.array(unique)-p,axis=1) if unique else np.array([])
        if len(distances) and distances.min()<radius:groups.append(int(distances.argmin()))
        else:groups.append(len(unique));unique.append(p)
    return np.array(unique).reshape(-1,2),groups

def triangles(points):
    ids=np.array(list(combinations(range(len(points)),3)),dtype=int).reshape(-1,3)
    if not len(ids):return ids,np.empty((0,2))
    p=points[ids]
    edges=np.stack([np.linalg.norm(p[:,1]-p[:,2],axis=1),np.linalg.norm(p[:,0]-p[:,2],axis=1),np.linalg.norm(p[:,0]-p[:,1],axis=1)],axis=1)
    order=np.argsort(edges,axis=1,kind='stable')
    ids=np.take_along_axis(ids,order,axis=1);edges=np.sort(edges,axis=1)
    p=points[ids];v=p[:,1]-p[:,0];w=p[:,2]-p[:,0]
    area=abs(v[:,0]*w[:,1]-v[:,1]*w[:,0])
    keep=(edges[:,0]>1e-5)&(area>0.02*edges[:,2]**2)
    return ids[keep],edges[keep,:2]/np.maximum(edges[keep,2,None],1e-6)

def assignment(transformed,points,tolerance,tags=None):
    d=np.linalg.norm(transformed[:,None,:]-points[None,:,:],axis=2)
    candidates=np.argwhere(d<tolerance)
    if not len(candidates):return [],[]
    order=np.argsort(d[candidates[:,0],candidates[:,1]],kind='stable')
    a=set();b=set();pairs=[];res=[]
    for k in order:
        i,j=candidates[k]
        tag=int(tags[j]) if tags is not None else int(j)
        if i not in a and tag not in b:
            a.add(i);b.add(tag);pairs.append((int(i),int(j)));res.append(float(d[i,j]))
    return pairs,res

def recognize(points,patterns,seed=6643,cap=50000,tolerance=24.,alternatives=None,use_quads=False,shear_penalty=0.,auxiliary_map=None):
    pts,groups=consolidate(np.asarray(points,dtype=float).reshape(-1,2))
    if len(pts)<3:return 'unknown',[],{'reason':'fewer than three unique points','groups':groups,'hypotheses':[]}
    scene_ids,scene_desc=triangles(pts)
    if not len(scene_ids):return 'unknown',[],{'reason':'degenerate scene','groups':groups,'hypotheses':[]}
    pool=pts;tags=None;pool_scores=None
    if alternatives is not None:
        pool=[];tags=[];pool_scores=[]
        for i,qs in enumerate(alternatives):
            for q in qs[:5]:
                if q[2]>=qs[0][2]-.10:
                    pool.append(q[:2]);tags.append(groups[i]);pool_scores.append(q[2])
        pool=np.asarray(pool);tags=np.asarray(tags);pool_scores=np.asarray(pool_scores)
    tree=cKDTree(scene_desc);hypotheses=[]
    per_class=cap//max(len(patterns),1)
    for name,template in sorted(patterns.items()):
        if len(template)<3:
            hypotheses.append({'name':name,'score':-1e9,'support':0,'reason':'pair-only ambiguity'});continue
        # Normalize schematic axis lengths because supplied canvas aspect is arbitrary.
        p=(template-template.mean(axis=0))/np.maximum(np.ptp(template,axis=0),1e-6)
        ti,td=triangles(p)
        if not len(ti):continue
        dist,near=tree.query(td,k=min(20,len(scene_desc)))
        if near.ndim==1:near=near[:,None];dist=dist[:,None]
        jobs=[(i,int(j),float(d)) for i,(ns,ds) in enumerate(zip(near,dist)) for j,d in zip(np.atleast_1d(ns),np.atleast_1d(ds))]
        # Deterministic class-local selection is invariant to catalog order.
        rng=np.random.default_rng(seed+int(hashlib.sha256(name.encode()).hexdigest()[:8],16))
        jobs.sort(key=lambda j:j[2])
        if len(jobs)>per_class:
            first=per_class//2
            ids=rng.choice(np.arange(first,len(jobs)),per_class-first,replace=False)
            jobs=jobs[:first]+[jobs[i] for i in ids]
        if use_quads:
            qjobs=quad_jobs(p,pts,per_class)
            fitting=[(p[t[:3]],pts[u[:3]]) for _,t,u in qjobs]
            fitting += [(p[ti[i]],pts[scene_ids[j]]) for i,j,_ in jobs[:per_class-len(fitting)]]
        else:
            fitting=[(p[ti[i]],pts[scene_ids[j]]) for i,j,_ in jobs]
        best={'name':name,'score':-1e9,'support':0};accepted=0
        for src,dst in fitting:
            a=np.c_[src,np.ones(3)]
            try:matrix=np.linalg.solve(a,dst)
            except np.linalg.LinAlgError:continue
            sv=np.linalg.svd(matrix[:2],compute_uv=False)
            if sv[-1]<50 or sv[0]>6000 or sv[0]/sv[-1]>8:continue
            accepted+=1;mapped=np.c_[p,np.ones(len(p))]@matrix
            pairs,res=assignment(mapped,pool,max(24.,tolerance),tags)
            if len(pairs)>=4:
                for _ in range(2):
                    ii,jj=np.array(pairs).T
                    matrix=np.linalg.lstsq(np.c_[p[ii],np.ones(len(ii))],pool[jj],rcond=None)[0]
                    mapped=np.c_[p,np.ones(len(p))]@matrix
                    pairs,res=assignment(mapped,pool,tolerance,tags)
                    if len(pairs)<4:break
            support=len(pairs)
            if support<4:continue
            # Fitting three nodes alone is never counted as class evidence.
            fraction=min(.8,len(pool)*np.pi*tolerance*tolerance/9e6)
            surprise=-float(binom.logsf(support-4,max(len(p)-3,1),fraction))/np.log(10)
            shear=abs(matrix[0]@matrix[1])/max(np.linalg.norm(matrix[0])*np.linalg.norm(matrix[1]),1e-9)
            score=surprise-.5*np.mean(res)/tolerance-shear_penalty*shear
            auxiliary=0.
            if auxiliary_map is not None:
                inside=(mapped[:,0]>=0)&(mapped[:,1]>=0)&(mapped[:,0]<auxiliary_map.shape[1])&(mapped[:,1]<auxiliary_map.shape[0])
                supported_nodes={i for i,_ in pairs}
                values=[]
                for k,pt in enumerate(mapped):
                    if k in supported_nodes:continue
                    values.append(float(auxiliary_map[int(pt[1]),int(pt[0])]) if inside[k] else 0.)
                auxiliary=float(np.mean(values)) if values else .5
                score+=3.*(auxiliary-.5)
            if score>best['score']:
                best={'name':name,'score':score,'support':support,'coverage':support/len(p),'mean_residual':float(np.mean(res)),'shear':float(shear),'auxiliary':auxiliary,'pairs':pairs,'matrix':matrix.tolist(),'nodes':mapped.tolist(),'matched_points':[pool[j].tolist() for _,j in pairs]}
        best.update(attempted=len(jobs),accepted=accepted,cap_hit=len(jobs)>=per_class)
        hypotheses.append(best)
    hypotheses.sort(key=lambda h:(-h['score'],h['name']))
    if not hypotheses or hypotheses[0]['support']<4:
        return 'unknown',[],{'reason':'no independently verified affine fit','groups':groups,'hypotheses':hypotheses}
    winner=hypotheses[0];supported={int(tags[j]) if tags is not None else j for _,j in winner['pairs']}
    members=[i for i,g in enumerate(groups) if g in supported]
    return winner['name'],members,{'groups':groups,'hypotheses':hypotheses,'score_gap':winner['score']-hypotheses[1]['score'] if len(hypotheses)>1 else None}
'''
path=WORK/'constellation/geometry.py'
path.parent.mkdir(parents=True,exist_ok=True)
path.write_text(SOURCE['constellation/geometry.py'])

In [ ]:
# Embedded implementation: constellation/geometry_base.py
SOURCE['constellation/geometry_base.py'] = r'''"""Triangle-hash hypotheses with independent, one-to-one affine verification."""
import hashlib
from itertools import combinations
import numpy as np
from scipy.spatial import cKDTree
from scipy.stats import binom

def consolidate(points,radius=3.):
    # Anchor grouping avoids transitive chains. Radius is deliberately far below
    # scoring tolerance; close sources separated by >3 pixels stay distinct.
    unique=[];groups=[]
    for p in points:
        distances=np.linalg.norm(np.array(unique)-p,axis=1) if unique else np.array([])
        if len(distances) and distances.min()<radius:groups.append(int(distances.argmin()))
        else:groups.append(len(unique));unique.append(p)
    return np.array(unique).reshape(-1,2),groups

def triangles(points):
    ids=np.array(list(combinations(range(len(points)),3)),dtype=int).reshape(-1,3)
    if not len(ids):return ids,np.empty((0,2))
    p=points[ids]
    edges=np.stack([np.linalg.norm(p[:,1]-p[:,2],axis=1),np.linalg.norm(p[:,0]-p[:,2],axis=1),np.linalg.norm(p[:,0]-p[:,1],axis=1)],axis=1)
    order=np.argsort(edges,axis=1,kind='stable')
    ids=np.take_along_axis(ids,order,axis=1);edges=np.sort(edges,axis=1)
    p=points[ids];v=p[:,1]-p[:,0];w=p[:,2]-p[:,0]
    area=abs(v[:,0]*w[:,1]-v[:,1]*w[:,0])
    keep=(edges[:,0]>1e-5)&(area>0.02*edges[:,2]**2)
    return ids[keep],edges[keep,:2]/np.maximum(edges[keep,2,None],1e-6)

def assignment(transformed,points,tolerance):
    d=np.linalg.norm(transformed[:,None,:]-points[None,:,:],axis=2)
    candidates=np.argwhere(d<tolerance)
    if not len(candidates):return [],[]
    order=np.argsort(d[candidates[:,0],candidates[:,1]],kind='stable')
    a=set();b=set();pairs=[];res=[]
    for k in order:
        i,j=candidates[k]
        if i not in a and j not in b:
            a.add(i);b.add(j);pairs.append((int(i),int(j)));res.append(float(d[i,j]))
    return pairs,res

def recognize(points,patterns,seed=6643,cap=50000,tolerance=24.):
    pts,groups=consolidate(np.asarray(points,dtype=float).reshape(-1,2))
    if len(pts)<3:return 'unknown',[],{'reason':'fewer than three unique points','groups':groups,'hypotheses':[]}
    scene_ids,scene_desc=triangles(pts)
    if not len(scene_ids):return 'unknown',[],{'reason':'degenerate scene','groups':groups,'hypotheses':[]}
    tree=cKDTree(scene_desc);hypotheses=[]
    per_class=cap//max(len(patterns),1)
    for name,template in sorted(patterns.items()):
        if len(template)<3:
            hypotheses.append({'name':name,'score':-1e9,'support':0,'reason':'pair-only ambiguity'});continue
        # Normalize schematic axis lengths because supplied canvas aspect is arbitrary.
        p=(template-template.mean(axis=0))/np.maximum(np.ptp(template,axis=0),1e-6)
        ti,td=triangles(p)
        if not len(ti):continue
        dist,near=tree.query(td,k=min(20,len(scene_desc)))
        if near.ndim==1:near=near[:,None];dist=dist[:,None]
        jobs=[(i,int(j),float(d)) for i,(ns,ds) in enumerate(zip(near,dist)) for j,d in zip(np.atleast_1d(ns),np.atleast_1d(ds))]
        # Deterministic class-local selection is invariant to catalog order.
        rng=np.random.default_rng(seed+int(hashlib.sha256(name.encode()).hexdigest()[:8],16))
        jobs.sort(key=lambda j:j[2])
        if len(jobs)>per_class:
            first=per_class//2
            ids=rng.choice(np.arange(first,len(jobs)),per_class-first,replace=False)
            jobs=jobs[:first]+[jobs[i] for i in ids]
        best={'name':name,'score':-1e9,'support':0};accepted=0
        for i,j,_ in jobs:
            src=p[ti[i]];dst=pts[scene_ids[j]]
            a=np.c_[src,np.ones(3)]
            try:matrix=np.linalg.solve(a,dst)
            except np.linalg.LinAlgError:continue
            sv=np.linalg.svd(matrix[:2],compute_uv=False)
            if sv[-1]<50 or sv[0]>6000 or sv[0]/sv[-1]>8:continue
            accepted+=1;mapped=np.c_[p,np.ones(len(p))]@matrix
            pairs,res=assignment(mapped,pts,tolerance)
            support=len(pairs)
            if support<4:continue
            # Fitting three nodes alone is never counted as class evidence.
            fraction=min(.8,len(pts)*np.pi*tolerance*tolerance/9e6)
            surprise=-float(binom.logsf(support-4,max(len(p)-3,1),fraction))/np.log(10)
            score=surprise-.5*np.mean(res)/tolerance
            if score>best['score']:
                best={'name':name,'score':score,'support':support,'coverage':support/len(p),'mean_residual':float(np.mean(res)),'pairs':pairs,'matrix':matrix.tolist(),'nodes':mapped.tolist()}
        best.update(attempted=len(jobs),accepted=accepted,cap_hit=len(jobs)>=per_class)
        hypotheses.append(best)
    hypotheses.sort(key=lambda h:(-h['score'],h['name']))
    if not hypotheses or hypotheses[0]['support']<4:
        return 'unknown',[],{'reason':'no independently verified affine fit','groups':groups,'hypotheses':hypotheses}
    winner=hypotheses[0];supported={j for _,j in winner['pairs']}
    members=[i for i,g in enumerate(groups) if g in supported]
    return winner['name'],members,{'groups':groups,'hypotheses':hypotheses,'score_gap':winner['score']-hypotheses[1]['score'] if len(hypotheses)>1 else None}
'''
path=WORK/'constellation/geometry_base.py'
path.parent.mkdir(parents=True,exist_ok=True)
path.write_text(SOURCE['constellation/geometry_base.py'])

In [ ]:
# Embedded implementation: constellation/joint.py
SOURCE['constellation/joint.py'] = r'''"""Joint localization and recognition.

The frozen recognizer consumed one point per query (the top appearance
alternative) and returned only a class plus membership. Measured on the labelled
scenes, the correct location is present in the 20 retained alternatives for 90%
of present queries but ranks first for only 61% of *figure* queries, with a
median appearance gap of 0.031. Appearance alone cannot separate them.

This module verifies hypotheses against the whole alternative pool under
one-to-one constraints per query group, and reports which pool point each query
matched so the caller can adopt the geometrically supported coordinate. A
restricted similarity/reflection branch runs beside the affine branch so a
four-degree-of-freedom explanation is preferred when it fits comparably well.
"""

import numpy as np
from scipy.spatial import cKDTree
from scipy.stats import binom
from .geometry import consolidate, triangles
from .quad import quads

# Rejected transform envelope: scene pixels per normalized template unit.
MIN_SV, MAX_SV, MAX_ANISO = 50., 6000., 8.
SIMILARITY_BONUS = .35   # reward for explaining the data with fewer parameters
GEN_LIMIT = 56           # quad generation is O(n^4); cap the seeding point set


def build_pool(alternatives, groups, top_k=8, margin=.15, gap=.03):
    """Flatten per-query alternatives into a verification pool tagged by group.

    Only *ambiguous* queries contribute more than their top alternative. On the
    labelled scenes the rank-0/rank-1 appearance gap separates correctly located
    queries (median 0.136) from mislocated ones (median 0.009), so a query whose
    top alternative wins clearly is not offered up for geometric relocation.
    """
    pts, tags, scores, ranks = [], [], [], []
    for i, qs in enumerate(alternatives):
        if not len(qs):
            continue
        best = qs[0][2]
        ambiguous = len(qs) > 1 and (best - qs[1][2]) < gap
        limit = top_k if ambiguous else 1
        for r, q in enumerate(qs[:limit]):
            if q[2] < best - margin:
                break
            pts.append(q[:2]); tags.append(groups[i])
            scores.append(q[2]); ranks.append(r)
    return (np.asarray(pts, dtype=float).reshape(-1, 2), np.asarray(tags, dtype=int),
            np.asarray(scores, dtype=float), np.asarray(ranks, dtype=int))


def assignment(transformed, points, tolerance, tags):
    """Greedy nearest-first one-to-one match; at most one point per query group."""
    if not len(points):
        return [], []
    d = np.linalg.norm(transformed[:, None, :] - points[None, :, :], axis=2)
    candidates = np.argwhere(d < tolerance)
    if not len(candidates):
        return [], []
    order = np.argsort(d[candidates[:, 0], candidates[:, 1]], kind='stable')
    used_node, used_tag, pairs, res = set(), set(), [], []
    for k in order:
        i, j = candidates[k]
        tag = int(tags[j])
        if i not in used_node and tag not in used_tag:
            used_node.add(i); used_tag.add(tag)
            pairs.append((int(i), int(j))); res.append(float(d[i, j]))
    return pairs, res


def fit_similarity(src, dst, reflect=False):
    """Least-squares similarity (optionally reflected) returned as a 3x2 matrix."""
    s = src * np.array([-1., 1.]) if reflect else src
    sm, dm = s.mean(0), dst.mean(0)
    a, b = s - sm, dst - dm
    den = float(np.sum(a * a))
    if den < 1e-12:
        return None
    c = float(np.sum(a * b)) / den
    d = float(np.sum(a[:, 0] * b[:, 1] - a[:, 1] * b[:, 0])) / den
    r = np.array([[c, d], [-d, c]])
    if reflect:
        r = np.array([[-1., 0.], [0., 1.]]) @ r
    matrix = np.zeros((3, 2))
    matrix[:2] = r
    matrix[2] = dm - src.mean(0) @ r
    return matrix


def fit_affine(src, dst):
    try:
        return np.linalg.lstsq(np.c_[src, np.ones(len(src))], dst, rcond=None)[0]
    except np.linalg.LinAlgError:
        return None


def valid(matrix):
    if not np.isfinite(matrix).all():
        return False
    sv = np.linalg.svd(matrix[:2], compute_uv=False)
    return not (sv[-1] < MIN_SV or sv[0] > MAX_SV or sv[0] / max(sv[-1], 1e-9) > MAX_ANISO)


def generation_points(alternatives, groups, limit=GEN_LIMIT):
    """Seeding point set: highest-scoring rank-0 alternatives, capped for quads."""
    items = [(qs[0][2], qs[0][:2]) for qs in alternatives if len(qs)]
    if len(items) > limit:
        keep = sorted(range(len(items)), key=lambda i: -items[i][0])[:limit]
        keep.sort()
        items = [items[i] for i in keep]
    return np.asarray([xy for _, xy in items], dtype=float).reshape(-1, 2)


class SceneIndex:
    """Scene-side invariants and KD-trees, built once and reused for all classes."""

    def __init__(self, gen_pts, use_quads=True):
        self.tri_ids, tri_desc = triangles(gen_pts)
        self.tri_tree = cKDTree(tri_desc) if len(tri_desc) else None
        self.quad_ids, self.quad_tree = np.empty((0, 4), int), None
        if use_quads and len(gen_pts) >= 4:
            qi, qd = quads(gen_pts)
            if len(qd):
                self.quad_ids, self.quad_tree = qi, cKDTree(qd)

    @staticmethod
    def _query(tree, ids, desc, k):
        if tree is None or not len(desc):
            return []
        dist, idx = tree.query(desc, k=min(k, tree.n))
        dist, idx = np.atleast_2d(dist), np.atleast_2d(idx)
        if dist.shape[0] != len(desc):
            dist, idx = dist.T, idx.T
        out = [(float(d), i, ids[j])
               for i in range(len(desc)) for j, d in zip(idx[i], dist[i])]
        out.sort(key=lambda x: x[0])
        return out

    def seeds(self, template, budget, quad_share=.6):
        """Quad and triangle seeds are budgeted separately.

        Quad invariants are 4-dimensional and triangle invariants 2-dimensional,
        so their nearest-neighbour distances are not on a common scale. Merging
        and sorting them lets triangles crowd out the more discriminative quads.
        """
        quad_seeds = []
        if self.quad_tree is not None and len(template) >= 4:
            ti, td = quads(template)
            quad_seeds = [(d, ti[i], sid)
                          for d, i, sid in self._query(self.quad_tree, self.quad_ids, td, 4)]
        tri_seeds = []
        ti3, td3 = triangles(template)
        tri_seeds = [(d, ti3[i], sid)
                     for d, i, sid in self._query(self.tri_tree, self.tri_ids, td3, 20)]
        n_quad = min(len(quad_seeds), int(budget * quad_share))
        n_tri = min(len(tri_seeds), budget - n_quad)
        n_quad = min(len(quad_seeds), budget - n_tri)
        return quad_seeds[:n_quad] + tri_seeds[:n_tri]


def recognize_joint(alternatives, patterns, seed=6643, cap=80000, tolerance=18.,
                    top_k=8, margin=.15, shear_penalty=2., auxiliary_map=None,
                    models=('affine',), min_support=4,
                    appearance_weight=0., rank_weight=0., gap=.03, diag_top=8,
                    score_mode='binom', sigma=5., size_penalty=0., quad_share=0.,
                    aux_weight=3.):
    """Return (name, {query index: (x, y)}, diagnostics).

    Defaults match what `finalize.finalize_joint` passes in production, so calling
    this directly reproduces the shipped configuration. The similarity branch and
    quad seeding remain selectable but measured worse; see FINDINGS.md.
    """
    anchors = np.array([q[0][:2] for q in alternatives if len(q)]).reshape(-1, 2)
    if len(anchors) < 3:
        return 'unknown', {}, {'reason': 'fewer than three points', 'hypotheses': []}
    _, groups = consolidate(anchors)
    # groups is indexed over queries that have candidates; map back to query ids.
    live = [i for i, q in enumerate(alternatives) if len(q)]
    group_of = {q: groups[k] for k, q in enumerate(live)}
    pool, tags, pool_scores, pool_ranks = build_pool(alternatives, groups, top_k, margin, gap)
    gen_pts = generation_points(alternatives, groups)
    if len(gen_pts) < 3 or not len(pool):
        return 'unknown', {}, {'reason': 'insufficient points', 'hypotheses': []}

    index = SceneIndex(gen_pts)
    n_groups = len(set(tags.tolist()))
    hypotheses = []
    per_class = cap // max(len(patterns), 1)
    for name, template in sorted(patterns.items()):
        if len(template) < 3:
            hypotheses.append({'name': name, 'score': -1e9, 'support': 0,
                               'reason': 'pair-only ambiguity'})
            continue
        # Normalize schematic axis lengths; supplied canvas aspect is arbitrary.
        p = (template - template.mean(axis=0)) / np.maximum(np.ptp(template, axis=0), 1e-6)
        # Seed selection is deterministic and class-local, so it is invariant to
        # catalog order; `seed` is retained only for reporting.
        seeds = index.seeds(p, per_class, quad_share)
        if not seeds:
            hypotheses.append({'name': name, 'score': -1e9, 'support': 0, 'reason': 'no seeds'})
            continue
        best = {'name': name, 'score': -1e9, 'support': 0}
        accepted = 0
        for _, ti, si in seeds:
            src, dst = p[np.asarray(ti)], gen_pts[np.asarray(si)]
            for model in models:
                if model == 'affine':
                    if len(src) == 3:
                        try:
                            matrix = np.linalg.solve(np.c_[src, np.ones(3)], dst)
                        except np.linalg.LinAlgError:
                            continue
                    else:
                        matrix = fit_affine(src, dst)
                    bonus = 0.
                else:
                    matrix = fit_similarity(src, dst)
                    mirror = fit_similarity(src, dst, reflect=True)
                    if matrix is not None and mirror is not None:
                        a = np.c_[src, np.ones(len(src))]
                        if np.linalg.norm(a @ mirror - dst) < np.linalg.norm(a @ matrix - dst):
                            matrix = mirror
                    bonus = SIMILARITY_BONUS
                if matrix is None or not valid(matrix):
                    continue
                accepted += 1
                mapped = np.c_[p, np.ones(len(p))] @ matrix
                pairs, res = assignment(mapped, pool, max(tolerance, 24.), tags)
                if len(pairs) >= min_support:
                    for _ in range(3):
                        ii, jj = np.array(pairs).T
                        refit = (fit_affine(p[ii], pool[jj]) if model == 'affine'
                                 else fit_similarity(p[ii], pool[jj]))
                        if refit is None or not valid(refit):
                            break
                        matrix = refit
                        mapped = np.c_[p, np.ones(len(p))] @ matrix
                        pairs, res = assignment(mapped, pool, tolerance, tags)
                        if len(pairs) < min_support:
                            break
                support = len(pairs)
                if support < min_support:
                    continue
                shear = abs(matrix[0] @ matrix[1]) / max(
                    np.linalg.norm(matrix[0]) * np.linalg.norm(matrix[1]), 1e-9)
                jj = [j for _, j in pairs]
                appearance = float(np.mean(pool_scores[jj]))
                rank_cost = float(np.mean(pool_ranks[jj]))
                if score_mode == 'likelihood':
                    # Nearly every template can collect four chance matches from a
                    # cluttered pool, so support alone does not discriminate. A
                    # genuine correspondence lands within ~1px (measured: median
                    # 0.78px), whereas a chance match is spread over the whole
                    # tolerance disc. Score each matched node by the log ratio of
                    # a Gaussian inlier density to the uniform chance density, and
                    # charge a fixed cost per template node offered up for matching.
                    r = np.asarray(res)
                    gain = (np.log(tolerance * tolerance / (2 * sigma * sigma))
                            - r * r / (2 * sigma * sigma))
                    score = float(gain.sum()) - size_penalty * len(p)
                else:
                    fraction = min(.8, n_groups * np.pi * tolerance * tolerance / 9e6)
                    surprise = -float(binom.logsf(support - min_support,
                                                  max(len(p) - 3, 1), fraction)) / np.log(10)
                    score = surprise - .5 * np.mean(res) / tolerance
                score += (bonus - shear_penalty * shear
                          + appearance_weight * appearance - rank_weight * rank_cost)
                auxiliary = .5
                if auxiliary_map is not None:
                    h, w = auxiliary_map.shape
                    inside = ((mapped[:, 0] >= 0) & (mapped[:, 1] >= 0)
                              & (mapped[:, 0] < w) & (mapped[:, 1] < h))
                    supported = {i for i, _ in pairs}
                    vals = [float(auxiliary_map[int(pt[1]), int(pt[0])]) if inside[k] else 0.
                            for k, pt in enumerate(mapped) if k not in supported]
                    auxiliary = float(np.mean(vals)) if vals else .5
                    score += aux_weight * (auxiliary - .5)
                if score > best['score']:
                    best = {'name': name, 'score': float(score), 'support': support,
                            'model': model, 'coverage': support / len(p),
                            'mean_residual': float(np.mean(res)), 'shear': float(shear),
                            'auxiliary': auxiliary, 'appearance': appearance,
                            'rank_cost': rank_cost, 'matrix': matrix.tolist(),
                            'nodes': mapped.tolist(),
                            'pairs': [(int(i), int(j)) for i, j in pairs]}
        best.update(attempted=len(seeds), accepted=accepted, cap_hit=len(seeds) >= per_class)
        hypotheses.append(best)

    hypotheses.sort(key=lambda h: (-h['score'], h['name']))
    if not hypotheses or hypotheses[0]['support'] < min_support:
        return 'unknown', {}, {'reason': 'no verified fit',
                               'hypotheses': hypotheses[:diag_top], 'groups': groups}
    winner = hypotheses[0]
    by_tag = {int(tags[j]): pool[j] for _, j in winner['pairs']}
    chosen = {i: (float(by_tag[group_of[i]][0]), float(by_tag[group_of[i]][1]))
              for i in live if group_of[i] in by_tag}
    diag = {'hypotheses': hypotheses[:diag_top], 'groups': groups,
            'pool_size': int(len(pool)),
            'score_gap': (winner['score'] - hypotheses[1]['score']
                          if len(hypotheses) > 1 else None)}
    return winner['name'], chosen, diag
'''
path=WORK/'constellation/joint.py'
path.parent.mkdir(parents=True,exist_ok=True)
path.write_text(SOURCE['constellation/joint.py'])

In [ ]:
# Embedded implementation: constellation/pipeline.py
SOURCE['constellation/pipeline.py'] = r'''"""Classical baselines. Prediction never reads labels or derives classes from scene IDs."""
from dataclasses import dataclass, asdict
from pathlib import Path
import time
import cv2
import numpy as np
from .contracts import ScenePrediction

GEOMETRY_MODES = ('radial', 'harmonic', 'hybrid', 'final', 'joint')
HARMONIC_MODES = ('harmonic', 'hybrid', 'final', 'joint')
DENSE_MODES = ('hybrid', 'final', 'joint')
REFINE_MODES = ('final', 'joint')


@dataclass
class Config:
    mode: str = 'joint'
    threshold: float = .72
    alternatives: int = 20
    seed: int = 6643
    threads: int = 4
    # Joint stage: verification tolerance, ambiguity gate, alternatives per
    # ambiguous query, hypothesis budget, quad seeding share.
    tolerance: float = 18.
    gap: float = .03
    top_k: int = 8
    cap: int = 80000
    quad_share: float = 0.
    # Verification representation and retained-alternative count. All seven
    # labelled present queries with no candidate within 12px reach the proposal
    # set and are lost inside verify; its correct proposal ranks 0-1 before
    # suppression, so the losses are tail truncation and representation, not
    # retrieval. 'blur' and 20 reproduce the previous baseline exactly.
    verify_rep: str = 'blur'
    def verify_pair(self, image, patch):
        """Preprocessed (scene, patch) pair handed to `retrieval.verify`."""
        f = VERIFY_REPS[self.verify_rep]
        return f(image.astype(np.float32)), f(patch.astype(np.float32))


def _blur(x, sigma=.6):
    return cv2.GaussianBlur(x, (0, 0), sigma)


def _dog(x, lo=.8, hi=2.5):
    return cv2.GaussianBlur(x, (0, 0), lo) - cv2.GaussianBlur(x, (0, 0), hi)


VERIFY_REPS = {'blur': _blur, 'dog': _dog}

def preprocess(image):
    x = image.astype(np.float32)
    return x-cv2.GaussianBlur(x,(0,0),3)

def distinct_maxima(scores, count=20, radius=16):
    scores = scores.copy()
    out=[]
    for _ in range(count):
        _,v,_,(x,y)=cv2.minMaxLoc(scores)
        if not np.isfinite(v): break
        out.append((x,y,float(v)))
        scores[max(0,y-radius):y+radius+1,max(0,x-radius):x+radius+1]=-np.inf
    return out

def predict_scene(image, patches, patterns, config):
    cv2.setNumThreads(config.threads)
    start=time.perf_counter()
    sky=preprocess(image)
    results=[]; diagnostics=[]
    if config.mode in GEOMETRY_MODES:
        from .retrieval import build_index,retrieve,verify
        if config.mode in HARMONIC_MODES:
            from .retrieval import build_harmonic_index as build_index, retrieve_harmonic as retrieve
        points,descriptors,stars=build_index(image)
    for patch in patches:
        q=preprocess(patch)
        if config.mode in GEOMETRY_MODES:
            proposed=retrieve(patch,points,descriptors)
            if config.mode in DENSE_MODES:
                from .dense import dense_candidates
                proposed=np.unique(np.vstack([proposed,dense_candidates(image,patch)]),axis=0)
            scene_rep,patch_rep=config.verify_pair(image,patch)
            refined=verify(scene_rep,patch_rep,proposed,config.alternatives)
            coarse=refined
            if config.mode in REFINE_MODES:
                from .refine import refine_candidates
                refined=refine_candidates(image,patch,refined)
            x,y,s,*_=refined[0]
            results.append((x,y,0) if s>=config.threshold else None)
            diagnostics.append({"candidates":refined,"appearance_score":s,"coarse_candidates":coarse})
            continue
        score=cv2.matchTemplate(sky,q,cv2.TM_CCOEFF_NORMED)
        candidates=distinct_maxima(score,config.alternatives)
        # OpenCV template locations are top-left corners. Pixel-center midpoint
        # of an even 32-pixel array is 15.5; the <=12-pixel metric is insensitive
        # to the unresolved dataset convention's possible half-pixel offset.
        candidates=[(x+(patch.shape[1]-1)/2,y+(patch.shape[0]-1)/2,s) for x,y,s in candidates]
        x,y,s=candidates[0]
        results.append((x,y,0) if s>=config.threshold else None)
        diagnostics.append({'candidates':candidates,'appearance_score':s})
    if config.mode in REFINE_MODES:
        from .references import extract_patterns
        refined_lists=[q['candidates'] for q in diagnostics]
        coarse_lists=[q['coarse_candidates'] for q in diagnostics]
        references=extract_patterns(patterns)
        if config.mode == 'joint':
            from .finalize import finalize_joint
            prediction=finalize_joint(image,refined_lists,coarse_lists,references,
                                      config.threshold,config.seed,tolerance=config.tolerance,
                                      gap=config.gap,top_k=config.top_k,cap=config.cap,
                                      quad_share=config.quad_share)
        else:
            from .finalize import finalize
            prediction=finalize(image,refined_lists,coarse_lists,references,config.threshold,config.seed)
        prediction.diagnostics.update(queries=diagnostics,runtime_seconds=time.perf_counter()-start,config=asdict(config),stage=config.mode)
        return prediction
    from .references import extract_patterns
    from .geometry import recognize
    present_ids=[i for i,p in enumerate(results) if p is not None]
    label,members,geometry=recognize([results[i][:2] for i in present_ids],extract_patterns(patterns),config.seed)
    for j in members:
        i=present_ids[j];x,y,_=results[i];results[i]=(x,y,1)
    return ScenePrediction(results,label,{'queries':diagnostics,'runtime_seconds':time.perf_counter()-start,'config':asdict(config),'stage':config.mode,'geometry':geometry})
'''
path=WORK/'constellation/pipeline.py'
path.parent.mkdir(parents=True,exist_ok=True)
path.write_text(SOURCE['constellation/pipeline.py'])

In [ ]:
# Embedded implementation: constellation/quad.py
SOURCE['constellation/quad.py'] = r'''from itertools import combinations
import numpy as np
from scipy.spatial import cKDTree

def quads(points):
    points=np.asarray(points,dtype=float)
    ids=np.array(list(combinations(range(len(points)),4)),dtype=np.int32).reshape(-1,4)
    if not len(ids):return ids,np.empty((0,4))
    p=points[ids];areas=[]
    for i in range(4):
        q=np.delete(p,i,axis=1);u=q[:,1]-q[:,0];v=q[:,2]-q[:,0]
        areas.append((u[:,0]*v[:,1]-u[:,1]*v[:,0])*(-1)**i)
    a=np.stack(areas,axis=1)
    largest=np.argmax(abs(a),axis=1);a*=np.where(a[np.arange(len(a)),largest]>=0,1.,-1.)[:,None]
    a/=np.maximum(abs(a).sum(axis=1,keepdims=True),1e-12)
    order=np.argsort(a,axis=1,kind='stable');a=np.take_along_axis(a,order,axis=1);ids=np.take_along_axis(ids,order,axis=1)
    keep=(np.min(abs(a),axis=1)>.015)&(np.min(np.diff(a,axis=1),axis=1)>.002)
    return ids[keep],a[keep]

def quad_jobs(template,points,budget):
    si,sd=quads(points);ti,td=quads(template)
    if not len(sd) or not len(td):return []
    distance,index=cKDTree(sd).query(td,k=min(4,len(sd)))
    if index.ndim==1:index=index[:,None];distance=distance[:,None]
    jobs=[]
    for i in range(len(ti)):
        for j,d in zip(index[i],distance[i]):jobs.append((float(d),ti[i],si[j]))
    jobs.sort(key=lambda x:x[0]);return jobs[:budget]
'''
path=WORK/'constellation/quad.py'
path.parent.mkdir(parents=True,exist_ok=True)
path.write_text(SOURCE['constellation/quad.py'])

In [ ]:
# Embedded implementation: constellation/references.py
SOURCE['constellation/references.py'] = r'''from pathlib import Path
import cv2
import numpy as np

def extract_patterns(folder):
    result={}
    for path in sorted(Path(folder).glob('*_pattern.png')):
        rgba=cv2.imread(str(path),cv2.IMREAD_UNCHANGED)
        # Supplied diagrams encode stars as opaque white disks and edges as green.
        mask=((rgba[:,:,:3].min(axis=2)>190)&(rgba[:,:,3]>100)).astype(np.uint8)
        n,labels,stats,centers=cv2.connectedComponentsWithStats(mask)
        points=centers[1:][stats[1:,cv2.CC_STAT_AREA]>=2]
        result[path.stem.removesuffix('_pattern')]=points
    return result
'''
path=WORK/'constellation/references.py'
path.parent.mkdir(parents=True,exist_ok=True)
path.write_text(SOURCE['constellation/references.py'])

In [ ]:
# Embedded implementation: constellation/refine.py
SOURCE['constellation/refine.py'] = r'''import cv2
import numpy as np

def refine_candidates(image,patch,candidates):
    q=patch.astype(np.float32)
    q=cv2.GaussianBlur(q,(0,0),.8)
    q=q-cv2.GaussianBlur(q,(0,0),3)
    yy,xx=np.mgrid[:32,:32].astype(np.float32)
    mask=((xx-15.5)**2+(yy-15.5)**2<14**2).astype(np.uint8)*255
    out=[]
    for x,y,score,angle,scale in candidates:
        crop=cv2.getRectSubPix(image,(64,64),(float(x),float(y))).astype(np.float32)
        crop=cv2.GaussianBlur(crop,(0,0),.8)
        crop=crop-cv2.GaussianBlur(crop,(0,0),3*scale)
        a=np.deg2rad(angle);c=scale*np.cos(a);s=scale*np.sin(a)
        warp=np.array([[c,s,31.5-15.5*(c+s)],[-s,c,31.5-15.5*(c-s)]],np.float32)
        original=warp.copy()
        try:
            cc,warp=cv2.findTransformECC(q,crop,warp,cv2.MOTION_AFFINE,(cv2.TERM_CRITERIA_COUNT|cv2.TERM_CRITERIA_EPS,40,1e-4),None,3)
            center=warp@np.array([15.5,15.5,1],np.float32)
            sv=np.linalg.svd(warp[:,:2],compute_uv=False)
            if np.linalg.norm(center-31.5)>5 or min(sv)<.65 or max(sv)>1.6 or max(sv)/min(sv)>1.35:warp=original
        except cv2.error:warp=original
        sampled=cv2.warpAffine(crop,warp,(32,32),flags=cv2.INTER_LINEAR|cv2.WARP_INVERSE_MAP)
        v=q[mask>0];w=sampled[mask>0];v=v-v.mean();w=w-w.mean()
        corr=float(v@w/max(np.linalg.norm(v)*np.linalg.norm(w),1e-6))
        center=warp@np.array([15.5,15.5,1],np.float32)
        out.append((float(x+center[0]-31.5),float(y+center[1]-31.5),corr,float(angle),float(scale)))
    return sorted(out,key=lambda p:-p[2])
'''
path=WORK/'constellation/refine.py'
path.parent.mkdir(parents=True,exist_ok=True)
path.write_text(SOURCE['constellation/refine.py'])

In [ ]:
# Embedded implementation: constellation/retrieval.py
SOURCE['constellation/retrieval.py'] = r'''"""Rotation-invariant radial proposals followed by masked rotation/scale verification."""
import cv2
import numpy as np
from scipy.spatial import cKDTree

def normalize(x):
    x=x-x.mean(axis=-1,keepdims=True)
    return x/np.maximum(np.linalg.norm(x,axis=-1,keepdims=True),1e-6)

def radial_maps(image):
    yy,xx=np.mgrid[-20:21,-20:21];r=np.hypot(xx,yy)
    maps=[]
    for radius in np.arange(0,19,2):
        kernel=np.exp(-.5*((r-radius)/1.15)**2).astype(np.float32);kernel/=kernel.sum()
        maps.append(cv2.filter2D(image,-1,kernel))
    return np.stack(maps,axis=-1)

def build_index(image,stride=4):
    raw=image.astype(np.float32)
    maps=radial_maps(raw)
    yy,xx=np.mgrid[20:raw.shape[0]-20:stride,20:raw.shape[1]-20:stride]
    dense=np.c_[xx.ravel(),yy.ravel()]
    blur=cv2.GaussianBlur(raw,(0,0),.8)
    dog=blur-cv2.GaussianBlur(raw,(0,0),2.5)
    maxima=(dog==cv2.dilate(dog,np.ones((5,5),np.uint8)))&(dog>1.5)
    y,x=np.where(maxima);keep=(x>=20)&(x<raw.shape[1]-20)&(y>=20)&(y<raw.shape[0]-20)
    stars=np.c_[x[keep],y[keep]]
    points=np.unique(np.vstack([dense,stars]),axis=0)
    desc=normalize(maps[points[:,1],points[:,0]])
    return points,desc,stars

def query_descriptors(patch):
    theta=np.arange(64)*2*np.pi/64
    out=[]
    for scale in (.75,.87,1.,1.15,1.33):
        radius=np.arange(0,19,2)/scale
        mx=(15.5+radius[:,None]*np.cos(theta)).astype(np.float32)
        my=(15.5+radius[:,None]*np.sin(theta)).astype(np.float32)
        values=cv2.remap(patch.astype(np.float32),mx,my,cv2.INTER_LINEAR,borderMode=cv2.BORDER_REFLECT_101)
        out.append(values.mean(axis=1))
    return normalize(np.array(out))

def retrieve(patch,points,descriptors,budget=1000):
    d=query_descriptors(patch)
    scores=np.max(descriptors@d.T,axis=1)
    ids=np.argpartition(scores,-budget)[-budget:]
    return points[ids[np.argsort(scores[ids])[::-1]]]

def verify(image,patch,candidates,keep=20):
    # Sample each sky neighborhood and compare to transformed query over a
    # common circular support; float32 maps avoid OpenCV sampling dtype errors.
    yy,xx=np.mgrid[-12:13:2,-12:13:2].astype(np.float32)
    mask=(xx*xx+yy*yy<=144);xx=xx[mask];yy=yy[mask]
    xmap=candidates[:,0,None].astype(np.float32)+xx
    ymap=candidates[:,1,None].astype(np.float32)+yy
    samples=cv2.remap(image,xmap,ymap,cv2.INTER_LINEAR)
    samples=normalize(samples)
    templates=[];transforms=[]
    for scale in (.75,.87,1.,1.15,1.33):
        for angle in np.arange(0,360,15):
            a=np.deg2rad(angle)
            mx=(15.5+(np.cos(a)*xx-np.sin(a)*yy)/scale).astype(np.float32)[None,:]
            my=(15.5+(np.sin(a)*xx+np.cos(a)*yy)/scale).astype(np.float32)[None,:]
            valid=(mx>=0)&(mx<=31)&(my>=0)&(my<=31)
            if not valid.all():continue
            v=cv2.remap(patch,mx,my,cv2.INTER_LINEAR).ravel()
            templates.append(v);transforms.append((float(angle),float(scale)))
    templates=normalize(np.array(templates))
    scores=samples@templates.T
    best=scores.max(axis=1); order=np.argsort(best)[::-1]
    selected=[]
    for idx in order:
        pt=candidates[idx]
        if any(np.linalg.norm(pt-np.array(s[:2]))<8 for s in selected):continue
        angle,scale=transforms[int(scores[idx].argmax())]
        selected.append((float(pt[0]),float(pt[1]),float(best[idx]),angle,scale))
        if len(selected)>=keep:break
    # Refine each alternative around its immutable coarse pose.
    refined=[]
    for x,y,_,angle,scale in selected:
        best=(-2,None)
        for da in (-7.5,0,7.5):
            a=np.deg2rad(angle+da)
            for ds in (.94,1,1.06):
                mx=(15.5+(np.cos(a)*xx-np.sin(a)*yy)/(scale*ds)).astype(np.float32)[None,:]
                my=(15.5+(np.sin(a)*xx+np.cos(a)*yy)/(scale*ds)).astype(np.float32)[None,:]
                if mx.min()<0 or mx.max()>31 or my.min()<0 or my.max()>31:continue
                q=normalize(cv2.remap(patch,mx,my,cv2.INTER_LINEAR))
                offsets=np.array([(dx,dy) for dx in (-2,-1,0,1,2) for dy in (-2,-1,0,1,2)],np.float32)
                sm=cv2.remap(image,(x+offsets[:,0,None]+xx).astype(np.float32),(y+offsets[:,1,None]+yy).astype(np.float32),cv2.INTER_LINEAR)
                corr=normalize(sm)@q.ravel();idx=int(corr.argmax())
                if corr[idx]>best[0]:best=(float(corr[idx]),(x+float(offsets[idx,0]),y+float(offsets[idx,1]),float(corr[idx]),angle+da,scale*ds))
        if best[1] is not None:refined.append(best[1])
    return sorted(refined,key=lambda x:-x[2])

def harmonic_features(image,points,scale=1.):
    """Circular harmonic magnitudes, invariant to in-plane rotation."""
    size=int(np.ceil(17*scale)); yy,xx=np.mgrid[-size:size+1,-size:size+1]
    radius=np.hypot(xx,yy)/scale;theta=np.arctan2(yy,xx)
    features=[]
    for r in (2,5,8,11,14):
        ring=np.exp(-.5*((radius-r)/1.2)**2);ring/=ring.sum()
        for order in (0,1,2,3):
            real=cv2.filter2D(image,-1,(ring*np.cos(order*theta)).astype(np.float32))
            rv=real[points[:,1],points[:,0]]
            if order:
                imag=cv2.filter2D(image,-1,(ring*np.sin(order*theta)).astype(np.float32))
                iv=imag[points[:,1],points[:,0]]
                features.append(np.hypot(rv,iv))
            else:features.append(rv)
    feat=np.stack(features,axis=-1)
    feat[:,::4]-=feat[:,::4].mean(axis=1,keepdims=True)
    return feat/np.maximum(np.linalg.norm(feat,axis=1,keepdims=True),1e-6)

def build_harmonic_index(image,stride=4):
    raw=cv2.GaussianBlur(image.astype(np.float32),(0,0),.6)
    yy,xx=np.mgrid[20:raw.shape[0]-20:stride,20:raw.shape[1]-20:stride]
    dense=np.c_[xx.ravel(),yy.ravel()]
    dog=raw-cv2.GaussianBlur(raw,(0,0),2.5)
    maxima=(dog==cv2.dilate(dog,np.ones((5,5),np.uint8)))&(dog>1.5)
    y,x=np.where(maxima);keep=(x>=20)&(x<raw.shape[1]-20)&(y>=20)&(y<raw.shape[0]-20)
    stars=np.c_[x[keep],y[keep]]
    points=np.unique(np.vstack([dense,stars]),axis=0)
    return points,harmonic_features(raw,points),stars

def retrieve_harmonic(patch,points,descriptors,budget=2000):
    raw=cv2.GaussianBlur(patch.astype(np.float32),(0,0),.6)
    ds=np.concatenate([harmonic_features(raw,np.array([[15,15],[16,16],[15,16],[16,15]]),scale) for scale in (.75,.87,1.,1.15,1.33)])
    scores=np.max(descriptors@ds.T,axis=1)
    ids=np.argpartition(scores,-budget)[-budget:]
    return points[ids[np.argsort(scores[ids])[::-1]]]
'''
path=WORK/'constellation/retrieval.py'
path.parent.mkdir(parents=True,exist_ok=True)
path.write_text(SOURCE['constellation/retrieval.py'])

In [ ]:
# Embedded implementation: run.py
SOURCE['run.py'] = r'''#!/usr/bin/env python3
"""Reproducible classical inference, evaluation and submission entrypoint."""
import argparse
from concurrent.futures import ProcessPoolExecutor,as_completed
from dataclasses import asdict
import csv,hashlib,json,platform,resource
from pathlib import Path
import cv2
import numpy as np
from constellation.contracts import read_truth,evaluate,write_submission
from constellation.pipeline import Config,predict_scene

def process_row(root,split,row,config,smoke,output):
    scene=Path(root)/split/row['Id']
    images=list(scene.glob('*_image.png'))
    if len(images)!=1:raise ValueError(f'Expected one sky in {scene}')
    image=cv2.imread(str(images[0]),0)
    n=int(row['n_patches']);n=min(n,2) if smoke else n
    patches=[cv2.imread(str(scene/'patches'/f'patch_{i:02}.png'),0) for i in range(1,n+1)]
    if image is None or any(q is None for q in patches):raise ValueError('Missing images')
    if image.shape!=(3000,3000) or any(q.shape!=(32,32) for q in patches):raise ValueError('Unexpected dimensions')
    prediction=predict_scene(image,patches,Path(root)/'patterns',config)
    prediction.diagnostics['peak_rss_platform_units']=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    (Path(output)/f"{row['Id']}.json").write_text(json.dumps(asdict(prediction),indent=2))
    print(row['Id'],round(prediction.diagnostics['runtime_seconds'],2),'seconds',flush=True)
    return row['Id'],prediction

def source_hash():
    here=Path(__file__).resolve().parent
    source=b''.join(f.read_bytes() for f in sorted((here/'constellation').glob('*.py')))+Path(__file__).read_bytes()
    return hashlib.sha256(source).hexdigest()

def main():
    p=argparse.ArgumentParser()
    p.add_argument('--data',type=Path,default=Path('.'))
    p.add_argument('--mode',choices=['smoke','evaluate','submission'],default='evaluate')
    p.add_argument('--output',type=Path,default=Path('outputs/final'))
    p.add_argument('--pipeline',choices=['a0','radial','harmonic','hybrid','final','joint'],default='joint')
    p.add_argument('--threshold',type=float,default=.72)
    p.add_argument('--workers',type=int,default=1)
    p.add_argument('--threads',type=int,default=2)
    p.add_argument('--verify-rep',choices=['blur','dog'],default='blur')
    p.add_argument('--alternatives',type=int,default=20)
    args=p.parse_args()
    if not 0<=args.threshold<=1 or args.workers<1:raise ValueError('Invalid configuration')
    root=args.data.resolve()
    if not (root/'patterns').exists() and (root/'participant').exists():root=root/'participant'
    args.output.mkdir(parents=True,exist_ok=True)
    config=Config(threshold=args.threshold,mode=args.pipeline,threads=args.threads,verify_rep=args.verify_rep,alternatives=args.alternatives)
    template=root/('sample_submission.csv' if args.mode=='submission' else 'train_ground_truth.csv')
    with template.open() as f:rows=list(csv.DictReader(f))
    if len({r['Id'] for r in rows})!=len(rows):raise ValueError('Duplicate IDs')
    if args.mode=='smoke':rows=rows[:1]
    digest=source_hash()
    manifest={'config':asdict(config),'python':platform.python_version(),'numpy':np.__version__,'opencv':cv2.__version__,'source_sha256':digest,'mode':args.mode,'workers':args.workers,'status':'running'}
    (args.output/'manifest.json').write_text(json.dumps(manifest,indent=2))
    split='validation' if args.mode=='submission' else 'train'
    predictions={}
    if args.workers==1:
        for row in rows:
            name,pred=process_row(root,split,row,config,args.mode=='smoke',args.output)
            predictions[name]=pred
    else:
        with ProcessPoolExecutor(max_workers=args.workers) as pool:
            futures=[pool.submit(process_row,root,split,row,config,args.mode=='smoke',args.output) for row in rows]
            for future in as_completed(futures):
                name,pred=future.result();predictions[name]=pred
    if source_hash()!=digest:raise RuntimeError('Source changed during inference; rerun from frozen source')
    if args.mode=='evaluate':
        metrics=evaluate(predictions,read_truth(template))
        (args.output/'metrics.json').write_text(json.dumps(metrics,indent=2));print(json.dumps(metrics,indent=2))
    if args.mode!='smoke':write_submission(predictions,template,args.output/'submission.csv')
    manifest.update(status='complete',peak_rss_platform_units=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss)
    (args.output/'manifest.json').write_text(json.dumps(manifest,indent=2))
if __name__=='__main__':main()
'''
path=WORK/'run.py'
path.parent.mkdir(parents=True,exist_ok=True)
path.write_text(SOURCE['run.py'])

In [ ]:
# Embedded implementation: audit.py
SOURCE['audit.py'] = r'''from pathlib import Path
import csv,hashlib,json
import cv2
import numpy as np
from PIL import Image,ImageDraw
from constellation.contracts import read_truth
out=Path('outputs/audit');out.mkdir(parents=True,exist_ok=True)
rows=[]; hashes={}; duplicates=[]
for path in sorted(Path('.').glob('**/*.png')):
    if path.parts[0] not in ('train','validation','patterns'):continue
    a=np.array(Image.open(path));h=hashlib.sha256(a.tobytes()).hexdigest()
    if h in hashes:duplicates.append([str(path),hashes[h]])
    hashes[h]=str(path)
    rows.append(dict(path=str(path),shape=str(a.shape),dtype=str(a.dtype),sha256=h))
with (out/'inventory.csv').open('w') as f:
    w=csv.DictWriter(f,fieldnames=rows[0]);w.writeheader();w.writerows(rows)
truth=read_truth('train_ground_truth.csv')
summary={n:dict(total=len(p.patches),figure=sum(x is not None and x[2]==1 for x in p.patches),off_figure=sum(x is not None and x[2]==0 for x in p.patches),absent=sum(x is None for x in p.patches)) for n,p in truth.items()}
(out/'summary.json').write_text(json.dumps(dict(files=len(rows),duplicates=duplicates,labels=summary),indent=2))
paths=sorted(Path('patterns').glob('*.png'))
canvas=Image.new('RGB',(1200,1600),'#202030');draw=ImageDraw.Draw(canvas)
for i,path in enumerate(paths):
    im=Image.open(path).convert('RGBA');bg=Image.new('RGBA',im.size,'white');bg.alpha_composite(im);bg=bg.convert('RGB');bg.thumbnail((185,170))
    x=(i%6)*200;y=(i//6)*200;canvas.paste(bg,(x,y+20));draw.text((x+2,y+2),path.stem.replace('_pattern',''),fill='white')
canvas.save(out/'patterns.jpg')
print(json.dumps(summary,indent=2));print('Files',len(rows),'duplicates',len(duplicates))
'''
path=WORK/'audit.py'
path.parent.mkdir(parents=True,exist_ok=True)
path.write_text(SOURCE['audit.py'])

In [ ]:
# Embedded implementation: calibrate.py
SOURCE['calibrate.py'] = r'''"""Leave-one-scene-out threshold calibration using saved blind candidates."""
import argparse,json
from pathlib import Path
import numpy as np
from constellation.contracts import ScenePrediction,evaluate,read_truth
from constellation.geometry import recognize
from constellation.references import extract_patterns

def prediction(saved,threshold,patterns=None):
    patches=[]
    for q in saved['diagnostics']['queries']:
        x,y,score,*_=q['candidates'][0]
        patches.append((x,y,0) if score>=threshold else None)
    label='unknown';d={}
    if patterns is not None:
        ids=[i for i,p in enumerate(patches) if p is not None]
        label,m,d=recognize([patches[i][:2] for i in ids],patterns)
        for j in m:
            i=ids[j];x,y,_=patches[i];patches[i]=(x,y,1)
    return ScenePrediction(patches,label,d)

def main():
    parser=argparse.ArgumentParser();parser.add_argument('--input',type=Path,required=True);args=parser.parse_args()
    truth=read_truth('train_ground_truth.csv');saved={n:json.loads((args.input/f'{n}.json').read_text()) for n in truth}
    thresholds=np.arange(.65,.991,.01);scores={};all_pred={}
    # Threshold fitting deliberately excludes identification to avoid rerunning
    # expensive geometry and to keep presence calibration appearance-driven.
    for threshold in thresholds:
        pred={n:prediction(s,float(threshold)) for n,s in saved.items()}
        scores[float(threshold)]=evaluate(pred,truth)
    folds={};held={};patterns=extract_patterns('patterns')
    for name in truth:
        dev=[n for n in truth if n!=name]
        threshold=max(scores,key=lambda t:np.mean([scores[t]['scenes'][n]['score'] for n in dev]))
        held[name]=prediction(saved[name],threshold,patterns)
        folds[name]={'development_scenes':dev,'threshold':threshold}
    chosen=max(scores,key=lambda t:scores[t]['mean']['score'])
    result={'folds':folds,'held_out':evaluate(held,truth),'final_threshold':chosen,'selection':'maximize development-scene mean of 0.25 presence + 0.20 localization + 0.25 recovery; geometry evaluated after threshold selection','limitation':'All three scenes informed iterative method development; LOSO threshold results are not an untouched estimate of method selection generalization.'}
    (args.input/'calibration.json').write_text(json.dumps(result,indent=2));print(json.dumps(result,indent=2))
if __name__=='__main__':main()
'''
path=WORK/'calibrate.py'
path.parent.mkdir(parents=True,exist_ok=True)
path.write_text(SOURCE['calibrate.py'])

In [ ]:
# Embedded implementation: geometry_oracle.py
SOURCE['geometry_oracle.py'] = r'''import json
from pathlib import Path
from constellation.contracts import read_truth
from constellation.references import extract_patterns
from constellation.geometry import recognize
truth=read_truth('train_ground_truth.csv');patterns=extract_patterns('patterns');out={}
print({n:len(p) for n,p in patterns.items()},flush=True)
for n,t in truth.items():
    out[n]={}
    for mode in ('present','figure'):
        points=[p[:2] for p in t.patches if p is not None and (mode=='present' or p[2]==1)]
        label,m,d=recognize(points,patterns)
        out[n][mode]={'prediction':label,'diagnostics':d}
        print(n,mode,label,[(h['name'],h['support'],round(h['score'],2)) for h in d['hypotheses'][:3]],flush=True)
Path('outputs/geometry_oracles.json').write_text(json.dumps(out,indent=2))
'''
path=WORK/'geometry_oracle.py'
path.parent.mkdir(parents=True,exist_ok=True)
path.write_text(SOURCE['geometry_oracle.py'])

In [ ]:
# Embedded implementation: requirements.txt
SOURCE['requirements.txt'] = r'''numpy>=2.0
scipy>=1.14
opencv-python-headless>=4.10
Pillow>=10
nbformat>=5.10
kaggle>=1.6
'''
path=WORK/'requirements.txt'
path.parent.mkdir(parents=True,exist_ok=True)
path.write_text(SOURCE['requirements.txt'])

## 2. Inventory and data contracts
Decode every supplied PNG, record dimensions/channels and decoded hashes, and count labelled categories. Ground truth is used only in evaluation and diagnostics.

In [ ]:
subprocess.check_call([sys.executable,str(WORK/'audit.py')],cwd=DATA)
print((DATA/'outputs/audit/summary.json').read_text())

## 3. Sky and patch statistics
The training counts and inventory above are measured. Absent patches are visually degraded in the same manner as present patches; they cannot be labelled from appearance alone.

In [ ]:
import numpy as np
from PIL import Image
rows=[]
for split in ['train','validation']:
    for p in sorted((DATA/split).glob('*/*_image.png')):
        a=np.array(Image.open(p));rows.append({'scene':p.parent.name,'mean':float(a.mean()),'std':float(a.std()),'min':int(a.min()),'max':int(a.max())})
print(json.dumps(rows,indent=2))

## 4. Reference extraction
White disks in the supplied RGBA diagrams encode stars; green lines encode edges. Extraction uses opaque white connected components, not scene names or manually labelled validation answers. Pair-only references are explicitly ambiguous.

In [ ]:
from constellation.references import extract_patterns
patterns=extract_patterns(DATA/'patterns')
print({name:len(points) for name,points in patterns.items()})

## 5. Candidate coverage, ambiguity and duplicate handling
The matcher unions dense harmonic retrieval with an exhaustive full-image rotation/scale search and keeps 20 spatial alternatives per query. On the labelled scenes the correct location is among those alternatives for 90% of present queries but ranks first for only 61% of figure queries, with a median appearance gap of 0.031 between the top two. That gap separates correctly located queries (median 0.136) from mislocated ones (median 0.009), so it is used to decide which queries are appearance-ambiguous and may be relocated by geometry. Every query retains an independent output. Geometry uses anchor-based grouping within three pixels; image-supported close-source fitting remains an outstanding plan item.

## 6. Classical localization and geometry
`a0`: translation baseline. `radial`: radial retrieval. `harmonic`: circular harmonic retrieval. `hybrid`: harmonic plus exhaustive search. `final`: hybrid plus ECC, quad+triangle affine hashing over one point per query. `joint`: the same candidates, but hypotheses are verified against the alternatives of ambiguous queries under one-to-one constraints, the winning fit supplies the reported coordinate, and seeding uses triangle invariants only because four-point quad invariants compound the measured ~5px template-to-sky model error.

In [ ]:
PIPELINE=os.environ.get('CONSTELLATION_PIPELINE','joint')
THRESHOLD=float(os.environ.get('CONSTELLATION_THRESHOLD','0.72'))
subprocess.check_call([sys.executable,str(WORK/'run.py'),'--data',str(DATA),'--mode',MODE,'--pipeline',PIPELINE,'--threshold',str(THRESHOLD),'--output',str(OUTPUT)])

## 7. Evaluation and limitations
Published score: 25% presence, 20% localization, 25% greedy recovery, 30% name. Empty-denominator conventions are unofficial. The three labelled scenes informed method development, so threshold holdouts do not constitute untouched method-selection validation. Geometry oracles must not be described as blind performance.

In [ ]:
for name in ['metrics.json','manifest.json']:
    p=OUTPUT/name
    if p.exists():print(name,p.read_text())

## 8. Submission and standalone export
The script below uses the identical source as this notebook. Submission mode preserves the sample CSV scene set, column order, query count and padding. Verify results before uploading. Smoke mode intentionally does not emit a submission.

In [ ]:
STANDALONE = '#!/usr/bin/env python3\n"""Self-contained source export. Install requirements, then use --help."""\nimport tempfile, pathlib, sys, runpy\nSOURCES = {\'constellation/__init__.py\': \'"""Scene-independent constellation detection."""\\n\', \'constellation/contracts.py\': "from dataclasses import dataclass, field\\nimport ast\\nimport csv\\nimport numpy as np\\n\\n@dataclass\\nclass ScenePrediction:\\n    patches: list\\n    constellation: str = \'unknown\'\\n    diagnostics: dict = field(default_factory=dict)\\n\\ndef parse_cell(value):\\n    if str(value).strip() == \'-1\':\\n        return None\\n    p = ast.literal_eval(value)\\n    if len(p) != 3 or p[2] not in (0, 1) or not np.isfinite(p).all():\\n        raise ValueError(f\'Invalid patch: {value}\')\\n    return tuple(p)\\n\\ndef read_truth(path):\\n    with open(path, newline=\'\') as f:\\n        rows = list(csv.DictReader(f))\\n    return {r[\'Id\']: ScenePrediction([parse_cell(r[f\'patch_{i:02}\']) for i in range(1, int(r[\'n_patches\'])+1)], r[\'constellation\']) for r in rows}\\n\\ndef reward(distance):\\n    return np.clip((36 - np.asarray(distance)) / 24, 0, 1)\\n\\ndef evaluate(predictions, ground_truth):\\n    results = {}\\n    for name, truth in ground_truth.items():\\n        pred = predictions[name]\\n        if len(pred.patches) != len(truth.patches):\\n            raise ValueError(\'Query count mismatch\')\\n        y = np.array([p is not None for p in truth.patches])\\n        z = np.array([p is not None for p in pred.patches])\\n        f1 = []\\n        for cls in (False, True):\\n            tp = np.sum((y == cls) & (z == cls))\\n            den = np.sum(y == cls) + np.sum(z == cls)\\n            f1.append(2*tp/den if den else 1.0)\\n        loc = [float(reward(np.linalg.norm(np.array(t[:2])-p[:2]))) if p is not None else 0.0 for p,t in zip(pred.patches, truth.patches) if t is not None]\\n        figure = np.array([t[:2] for t in truth.patches if t is not None and t[2] == 1]).reshape(-1,2)\\n        points = np.array([p[:2] for p in pred.patches if p is not None]).reshape(-1,2)\\n        total = 0.\\n        if len(figure) and len(points):\\n            d = np.linalg.norm(figure[:,None,:]-points[None,:,:], axis=2)\\n            used_f, used_p = set(), set()\\n            for flat in np.argsort(d, axis=None, kind=\'stable\'):\\n                i,j = np.unravel_index(flat, d.shape)\\n                if i not in used_f and j not in used_p:\\n                    total += float(reward(d[i,j]))\\n                    used_f.add(i); used_p.add(j)\\n        m = dict(presence=float(np.mean(f1)), localization=float(np.mean(loc)) if loc else 1., recovery=total/len(figure) if len(figure) else 1., identification=float(pred.constellation==truth.constellation))\\n        m[\'score\'] = sum(m[k]*w for k,w in [(\'presence\',.25),(\'localization\',.2),(\'recovery\',.25),(\'identification\',.3)])\\n        results[name] = m\\n    return {\'scenes\':results, \'mean\':{k:float(np.mean([r[k] for r in results.values()])) for k in next(iter(results.values()))}, \'worst_score\':min(r[\'score\'] for r in results.values()), \'conventions\':\'Unofficial: empty class F1 and empty localization/recovery = 1; greedy distance ties use stable row-major order.\'}\\n\\ndef write_submission(predictions, sample_submission, output_path):\\n    with open(sample_submission, newline=\'\') as f:\\n        reader = csv.DictReader(f); fields = reader.fieldnames; rows = list(reader)\\n    if set(predictions) != {r[\'Id\'] for r in rows}:\\n        raise ValueError(\'Scene set mismatch\')\\n    for row in rows:\\n        p = predictions[row[\'Id\']]\\n        if len(p.patches) != int(row[\'n_patches\']):\\n            raise ValueError(\'Query count mismatch\')\\n        for col in fields:\\n            if col.startswith(\'patch_\'):\\n                idx = int(col.split(\'_\')[1])-1\\n                point = p.patches[idx] if idx < len(p.patches) else None\\n                row[col] = \'-1\' if point is None else str((round(float(point[0]),3),round(float(point[1]),3),int(point[2])))\\n        row[\'constellation\'] = p.constellation\\n    with open(output_path,\'w\',newline=\'\') as f:\\n        writer = csv.DictWriter(f,fieldnames=fields); writer.writeheader(); writer.writerows(rows)\\n", \'constellation/dense.py\': \'"""Slower exhaustive coarse appearance fallback; no source detector dependency."""\\nimport cv2\\nimport numpy as np\\n\\ndef dense_candidates(image,patch,per_pose=3):\\n    # Downsample only scene search. Candidate poses are reverified at full resolution.\\n    small=cv2.resize(image.astype(np.float32),None,fx=.5,fy=.5,interpolation=cv2.INTER_AREA)\\n    small=small-cv2.GaussianBlur(small,(0,0),2.)\\n    q=patch.astype(np.float32)\\n    q=q-cv2.GaussianBlur(q,(0,0),4.)\\n    coords=np.arange(-7,8,dtype=np.float32)*2\\n    xx,yy=np.meshgrid(coords,coords)\\n    candidates=[]\\n    for scale in (.75,.87,1.,1.15,1.33):\\n        # Largest inscribed square valid under every rotation at this scale.\\n        half=int(np.floor(15*scale/np.sqrt(2)/2))\\n        yy,xx=np.mgrid[-half:half+1,-half:half+1].astype(np.float32)*2\\n        for angle in np.arange(0,360,15):\\n            a=np.deg2rad(angle)\\n            mx=(15.5+(np.cos(a)*xx-np.sin(a)*yy)/scale).astype(np.float32)\\n            my=(15.5+(np.sin(a)*xx+np.cos(a)*yy)/scale).astype(np.float32)\\n            template=cv2.remap(q,mx,my,cv2.INTER_LINEAR)\\n            scores=cv2.matchTemplate(small,template,cv2.TM_CCOEFF_NORMED)\\n            for _ in range(per_pose):\\n                _,v,_,(x,y)=cv2.minMaxLoc(scores)\\n                candidates.append(((x+half)*2+.5,(y+half)*2+.5))\\n                scores[max(0,y-8):y+9,max(0,x-8):x+9]=-1\\n    return np.unique(np.array(candidates,dtype=np.float32),axis=0)\\n\', \'constellation/finalize.py\': \'import cv2\\nimport numpy as np\\nfrom .geometry import recognize\\nfrom .joint import recognize_joint\\nfrom .contracts import ScenePrediction\\n\\n# Coarse-stage appearance scores are on a different scale from ECC-refined ones.\\nCOARSE_CUTOFF = .65\\n\\n\\ndef auxiliary_map(image):\\n    raw = image.astype(np.float32)\\n    dog = cv2.GaussianBlur(raw, (0, 0), 1) - cv2.GaussianBlur(raw, (0, 0), 8)\\n    response = cv2.dilate(dog, np.ones((25, 25), np.uint8))\\n    reference = np.sort(response.ravel()[::10])\\n    return (np.searchsorted(reference, response) / len(reference)).astype(np.float32)\\n\\n\\ndef finalize(image, queries, raw_queries, patterns, threshold=.72, seed=6643):\\n    """Frozen milestone stage: one point per query, quad+triangle affine hashing."""\\n    aux = auxiliary_map(image); competing = []\\n    for stage, qs, cutoff in [(\\\'refined\\\', queries, threshold),\\n                              (\\\'coarse\\\', raw_queries, COARSE_CUTOFF)]:\\n        ids = [i for i, q in enumerate(qs) if q[0][2] >= cutoff]\\n        name, m, d = recognize([qs[i][0][:2] for i in ids], patterns, seed=seed,\\n                               use_quads=True, shear_penalty=2., tolerance=18.,\\n                               auxiliary_map=aux)\\n        best = d[\\\'hypotheses\\\'][0] if d.get(\\\'hypotheses\\\') else {\\\'score\\\': -1e9, \\\'support\\\': 0}\\n        competing.append((best[\\\'score\\\'], name, d, stage))\\n    competing.sort(key=lambda x: (-x[0], x[1], x[3]))\\n    _, name, geometry, stage = competing[0]\\n    nodes = (np.array(geometry[\\\'hypotheses\\\'][0][\\\'nodes\\\'])\\n             if geometry.get(\\\'hypotheses\\\') and \\\'nodes\\\' in geometry[\\\'hypotheses\\\'][0]\\n             else np.empty((0, 2)))\\n    patches = []\\n    for q in queries:\\n        x, y, score, *_ = q[0]\\n        member = int(len(nodes) > 0 and np.linalg.norm(nodes - [x, y], axis=1).min() < 18)\\n        patches.append((x, y, member) if score >= threshold else None)\\n    return ScenePrediction(patches, name, {\\n        \\\'geometry\\\': geometry, \\\'selected_geometry_stage\\\': stage,\\n        \\\'competing_geometry\\\': [{\\\'stage\\\': s, \\\'name\\\': n, \\\'score\\\': float(v)}\\n                               for v, n, d, s in competing]})\\n\\n\\ndef finalize_joint(image, queries, raw_queries, patterns, threshold=.72, seed=6643,\\n                   tolerance=18., gap=.03, top_k=8, cap=80000, quad_share=0.,\\n                   member_radius=18., aux_weight=3., snap=True, snap_min_gap=0.):\\n    """Joint localization and recognition.\\n\\n    Differs from `finalize` in three measured ways. Verification runs against the\\n    alternatives of appearance-ambiguous queries rather than one point each, and\\n    the winning fit\\\'s chosen alternative replaces the reported coordinate. Seeding\\n    uses triangle invariants only, because four-point quad invariants compound the\\n    ~5px template-to-sky model error. The hypothesis budget is raised, which only\\n    matters once that model error is represented.\\n\\n    On 192 synthetic scenes at an unseen seed, identification is 0.474 against\\n    0.125 for `finalize`, and relocation produced 230 fixes with 0 regressions.\\n    On the three labelled scenes the weighted mean is 0.729 against 0.676.\\n    """\\n    aux = auxiliary_map(image)\\n    competing = []\\n    for stage, qs, cutoff in [(\\\'refined\\\', queries, threshold),\\n                              (\\\'coarse\\\', raw_queries, COARSE_CUTOFF)]:\\n        ids = [i for i, q in enumerate(qs) if len(q) and q[0][2] >= cutoff]\\n        if len(ids) < 3:\\n            continue\\n        name, chosen, d = recognize_joint(\\n            [qs[i] for i in ids], patterns, seed=seed, tolerance=tolerance, gap=gap,\\n            top_k=top_k, cap=cap, quad_share=quad_share, aux_weight=aux_weight,\\n            models=(\\\'affine\\\',), shear_penalty=2., auxiliary_map=aux)\\n        best = d[\\\'hypotheses\\\'][0] if d.get(\\\'hypotheses\\\') else {\\\'score\\\': -1e9}\\n        if not snap or (d.get(\\\'score_gap\\\') or 0.) < snap_min_gap:\\n            chosen = {}\\n        competing.append((best.get(\\\'score\\\', -1e9), name,\\n                          {ids[k]: v for k, v in chosen.items()}, d, stage))\\n    if not competing:\\n        return ScenePrediction([None] * len(queries), \\\'unknown\\\',\\n                               {\\\'reason\\\': \\\'too few candidate points\\\'})\\n    competing.sort(key=lambda x: (-x[0], x[1], x[4]))\\n    _, name, chosen, geometry, stage = competing[0]\\n    nodes = (np.array(geometry[\\\'hypotheses\\\'][0].get(\\\'nodes\\\', [])).reshape(-1, 2)\\n             if geometry.get(\\\'hypotheses\\\') else np.empty((0, 2)))\\n    patches = []\\n    for i, q in enumerate(queries):\\n        if not len(q):\\n            patches.append(None); continue\\n        x, y, score = float(q[0][0]), float(q[0][1]), float(q[0][2])\\n        if i in chosen:\\n            x, y = chosen[i]\\n        member = int(len(nodes) > 0\\n                     and np.linalg.norm(nodes - [x, y], axis=1).min() < member_radius)\\n        patches.append((x, y, member) if score >= threshold else None)\\n    return ScenePrediction(patches, name, {\\n        \\\'geometry\\\': geometry, \\\'selected_geometry_stage\\\': stage,\\n        \\\'relocated_queries\\\': sorted(chosen),\\n        \\\'competing_geometry\\\': [{\\\'stage\\\': s, \\\'name\\\': n, \\\'score\\\': float(v)}\\n                               for v, n, _, _, s in competing]})\\n\', \'constellation/geometry.py\': \'"""Triangle-hash hypotheses with independent, one-to-one affine verification."""\\nimport hashlib\\nfrom itertools import combinations\\nimport numpy as np\\nfrom scipy.spatial import cKDTree\\nfrom scipy.stats import binom\\nfrom .quad import quad_jobs\\n\\ndef consolidate(points,radius=3.):\\n    # Anchor grouping avoids transitive chains. Radius is deliberately far below\\n    # scoring tolerance; close sources separated by >3 pixels stay distinct.\\n    unique=[];groups=[]\\n    for p in points:\\n        distances=np.linalg.norm(np.array(unique)-p,axis=1) if unique else np.array([])\\n        if len(distances) and distances.min()<radius:groups.append(int(distances.argmin()))\\n        else:groups.append(len(unique));unique.append(p)\\n    return np.array(unique).reshape(-1,2),groups\\n\\ndef triangles(points):\\n    ids=np.array(list(combinations(range(len(points)),3)),dtype=int).reshape(-1,3)\\n    if not len(ids):return ids,np.empty((0,2))\\n    p=points[ids]\\n    edges=np.stack([np.linalg.norm(p[:,1]-p[:,2],axis=1),np.linalg.norm(p[:,0]-p[:,2],axis=1),np.linalg.norm(p[:,0]-p[:,1],axis=1)],axis=1)\\n    order=np.argsort(edges,axis=1,kind=\\\'stable\\\')\\n    ids=np.take_along_axis(ids,order,axis=1);edges=np.sort(edges,axis=1)\\n    p=points[ids];v=p[:,1]-p[:,0];w=p[:,2]-p[:,0]\\n    area=abs(v[:,0]*w[:,1]-v[:,1]*w[:,0])\\n    keep=(edges[:,0]>1e-5)&(area>0.02*edges[:,2]**2)\\n    return ids[keep],edges[keep,:2]/np.maximum(edges[keep,2,None],1e-6)\\n\\ndef assignment(transformed,points,tolerance,tags=None):\\n    d=np.linalg.norm(transformed[:,None,:]-points[None,:,:],axis=2)\\n    candidates=np.argwhere(d<tolerance)\\n    if not len(candidates):return [],[]\\n    order=np.argsort(d[candidates[:,0],candidates[:,1]],kind=\\\'stable\\\')\\n    a=set();b=set();pairs=[];res=[]\\n    for k in order:\\n        i,j=candidates[k]\\n        tag=int(tags[j]) if tags is not None else int(j)\\n        if i not in a and tag not in b:\\n            a.add(i);b.add(tag);pairs.append((int(i),int(j)));res.append(float(d[i,j]))\\n    return pairs,res\\n\\ndef recognize(points,patterns,seed=6643,cap=50000,tolerance=24.,alternatives=None,use_quads=False,shear_penalty=0.,auxiliary_map=None):\\n    pts,groups=consolidate(np.asarray(points,dtype=float).reshape(-1,2))\\n    if len(pts)<3:return \\\'unknown\\\',[],{\\\'reason\\\':\\\'fewer than three unique points\\\',\\\'groups\\\':groups,\\\'hypotheses\\\':[]}\\n    scene_ids,scene_desc=triangles(pts)\\n    if not len(scene_ids):return \\\'unknown\\\',[],{\\\'reason\\\':\\\'degenerate scene\\\',\\\'groups\\\':groups,\\\'hypotheses\\\':[]}\\n    pool=pts;tags=None;pool_scores=None\\n    if alternatives is not None:\\n        pool=[];tags=[];pool_scores=[]\\n        for i,qs in enumerate(alternatives):\\n            for q in qs[:5]:\\n                if q[2]>=qs[0][2]-.10:\\n                    pool.append(q[:2]);tags.append(groups[i]);pool_scores.append(q[2])\\n        pool=np.asarray(pool);tags=np.asarray(tags);pool_scores=np.asarray(pool_scores)\\n    tree=cKDTree(scene_desc);hypotheses=[]\\n    per_class=cap//max(len(patterns),1)\\n    for name,template in sorted(patterns.items()):\\n        if len(template)<3:\\n            hypotheses.append({\\\'name\\\':name,\\\'score\\\':-1e9,\\\'support\\\':0,\\\'reason\\\':\\\'pair-only ambiguity\\\'});continue\\n        # Normalize schematic axis lengths because supplied canvas aspect is arbitrary.\\n        p=(template-template.mean(axis=0))/np.maximum(np.ptp(template,axis=0),1e-6)\\n        ti,td=triangles(p)\\n        if not len(ti):continue\\n        dist,near=tree.query(td,k=min(20,len(scene_desc)))\\n        if near.ndim==1:near=near[:,None];dist=dist[:,None]\\n        jobs=[(i,int(j),float(d)) for i,(ns,ds) in enumerate(zip(near,dist)) for j,d in zip(np.atleast_1d(ns),np.atleast_1d(ds))]\\n        # Deterministic class-local selection is invariant to catalog order.\\n        rng=np.random.default_rng(seed+int(hashlib.sha256(name.encode()).hexdigest()[:8],16))\\n        jobs.sort(key=lambda j:j[2])\\n        if len(jobs)>per_class:\\n            first=per_class//2\\n            ids=rng.choice(np.arange(first,len(jobs)),per_class-first,replace=False)\\n            jobs=jobs[:first]+[jobs[i] for i in ids]\\n        if use_quads:\\n            qjobs=quad_jobs(p,pts,per_class)\\n            fitting=[(p[t[:3]],pts[u[:3]]) for _,t,u in qjobs]\\n            fitting += [(p[ti[i]],pts[scene_ids[j]]) for i,j,_ in jobs[:per_class-len(fitting)]]\\n        else:\\n            fitting=[(p[ti[i]],pts[scene_ids[j]]) for i,j,_ in jobs]\\n        best={\\\'name\\\':name,\\\'score\\\':-1e9,\\\'support\\\':0};accepted=0\\n        for src,dst in fitting:\\n            a=np.c_[src,np.ones(3)]\\n            try:matrix=np.linalg.solve(a,dst)\\n            except np.linalg.LinAlgError:continue\\n            sv=np.linalg.svd(matrix[:2],compute_uv=False)\\n            if sv[-1]<50 or sv[0]>6000 or sv[0]/sv[-1]>8:continue\\n            accepted+=1;mapped=np.c_[p,np.ones(len(p))]@matrix\\n            pairs,res=assignment(mapped,pool,max(24.,tolerance),tags)\\n            if len(pairs)>=4:\\n                for _ in range(2):\\n                    ii,jj=np.array(pairs).T\\n                    matrix=np.linalg.lstsq(np.c_[p[ii],np.ones(len(ii))],pool[jj],rcond=None)[0]\\n                    mapped=np.c_[p,np.ones(len(p))]@matrix\\n                    pairs,res=assignment(mapped,pool,tolerance,tags)\\n                    if len(pairs)<4:break\\n            support=len(pairs)\\n            if support<4:continue\\n            # Fitting three nodes alone is never counted as class evidence.\\n            fraction=min(.8,len(pool)*np.pi*tolerance*tolerance/9e6)\\n            surprise=-float(binom.logsf(support-4,max(len(p)-3,1),fraction))/np.log(10)\\n            shear=abs(matrix[0]@matrix[1])/max(np.linalg.norm(matrix[0])*np.linalg.norm(matrix[1]),1e-9)\\n            score=surprise-.5*np.mean(res)/tolerance-shear_penalty*shear\\n            auxiliary=0.\\n            if auxiliary_map is not None:\\n                inside=(mapped[:,0]>=0)&(mapped[:,1]>=0)&(mapped[:,0]<auxiliary_map.shape[1])&(mapped[:,1]<auxiliary_map.shape[0])\\n                supported_nodes={i for i,_ in pairs}\\n                values=[]\\n                for k,pt in enumerate(mapped):\\n                    if k in supported_nodes:continue\\n                    values.append(float(auxiliary_map[int(pt[1]),int(pt[0])]) if inside[k] else 0.)\\n                auxiliary=float(np.mean(values)) if values else .5\\n                score+=3.*(auxiliary-.5)\\n            if score>best[\\\'score\\\']:\\n                best={\\\'name\\\':name,\\\'score\\\':score,\\\'support\\\':support,\\\'coverage\\\':support/len(p),\\\'mean_residual\\\':float(np.mean(res)),\\\'shear\\\':float(shear),\\\'auxiliary\\\':auxiliary,\\\'pairs\\\':pairs,\\\'matrix\\\':matrix.tolist(),\\\'nodes\\\':mapped.tolist(),\\\'matched_points\\\':[pool[j].tolist() for _,j in pairs]}\\n        best.update(attempted=len(jobs),accepted=accepted,cap_hit=len(jobs)>=per_class)\\n        hypotheses.append(best)\\n    hypotheses.sort(key=lambda h:(-h[\\\'score\\\'],h[\\\'name\\\']))\\n    if not hypotheses or hypotheses[0][\\\'support\\\']<4:\\n        return \\\'unknown\\\',[],{\\\'reason\\\':\\\'no independently verified affine fit\\\',\\\'groups\\\':groups,\\\'hypotheses\\\':hypotheses}\\n    winner=hypotheses[0];supported={int(tags[j]) if tags is not None else j for _,j in winner[\\\'pairs\\\']}\\n    members=[i for i,g in enumerate(groups) if g in supported]\\n    return winner[\\\'name\\\'],members,{\\\'groups\\\':groups,\\\'hypotheses\\\':hypotheses,\\\'score_gap\\\':winner[\\\'score\\\']-hypotheses[1][\\\'score\\\'] if len(hypotheses)>1 else None}\\n\', \'constellation/geometry_base.py\': \'"""Triangle-hash hypotheses with independent, one-to-one affine verification."""\\nimport hashlib\\nfrom itertools import combinations\\nimport numpy as np\\nfrom scipy.spatial import cKDTree\\nfrom scipy.stats import binom\\n\\ndef consolidate(points,radius=3.):\\n    # Anchor grouping avoids transitive chains. Radius is deliberately far below\\n    # scoring tolerance; close sources separated by >3 pixels stay distinct.\\n    unique=[];groups=[]\\n    for p in points:\\n        distances=np.linalg.norm(np.array(unique)-p,axis=1) if unique else np.array([])\\n        if len(distances) and distances.min()<radius:groups.append(int(distances.argmin()))\\n        else:groups.append(len(unique));unique.append(p)\\n    return np.array(unique).reshape(-1,2),groups\\n\\ndef triangles(points):\\n    ids=np.array(list(combinations(range(len(points)),3)),dtype=int).reshape(-1,3)\\n    if not len(ids):return ids,np.empty((0,2))\\n    p=points[ids]\\n    edges=np.stack([np.linalg.norm(p[:,1]-p[:,2],axis=1),np.linalg.norm(p[:,0]-p[:,2],axis=1),np.linalg.norm(p[:,0]-p[:,1],axis=1)],axis=1)\\n    order=np.argsort(edges,axis=1,kind=\\\'stable\\\')\\n    ids=np.take_along_axis(ids,order,axis=1);edges=np.sort(edges,axis=1)\\n    p=points[ids];v=p[:,1]-p[:,0];w=p[:,2]-p[:,0]\\n    area=abs(v[:,0]*w[:,1]-v[:,1]*w[:,0])\\n    keep=(edges[:,0]>1e-5)&(area>0.02*edges[:,2]**2)\\n    return ids[keep],edges[keep,:2]/np.maximum(edges[keep,2,None],1e-6)\\n\\ndef assignment(transformed,points,tolerance):\\n    d=np.linalg.norm(transformed[:,None,:]-points[None,:,:],axis=2)\\n    candidates=np.argwhere(d<tolerance)\\n    if not len(candidates):return [],[]\\n    order=np.argsort(d[candidates[:,0],candidates[:,1]],kind=\\\'stable\\\')\\n    a=set();b=set();pairs=[];res=[]\\n    for k in order:\\n        i,j=candidates[k]\\n        if i not in a and j not in b:\\n            a.add(i);b.add(j);pairs.append((int(i),int(j)));res.append(float(d[i,j]))\\n    return pairs,res\\n\\ndef recognize(points,patterns,seed=6643,cap=50000,tolerance=24.):\\n    pts,groups=consolidate(np.asarray(points,dtype=float).reshape(-1,2))\\n    if len(pts)<3:return \\\'unknown\\\',[],{\\\'reason\\\':\\\'fewer than three unique points\\\',\\\'groups\\\':groups,\\\'hypotheses\\\':[]}\\n    scene_ids,scene_desc=triangles(pts)\\n    if not len(scene_ids):return \\\'unknown\\\',[],{\\\'reason\\\':\\\'degenerate scene\\\',\\\'groups\\\':groups,\\\'hypotheses\\\':[]}\\n    tree=cKDTree(scene_desc);hypotheses=[]\\n    per_class=cap//max(len(patterns),1)\\n    for name,template in sorted(patterns.items()):\\n        if len(template)<3:\\n            hypotheses.append({\\\'name\\\':name,\\\'score\\\':-1e9,\\\'support\\\':0,\\\'reason\\\':\\\'pair-only ambiguity\\\'});continue\\n        # Normalize schematic axis lengths because supplied canvas aspect is arbitrary.\\n        p=(template-template.mean(axis=0))/np.maximum(np.ptp(template,axis=0),1e-6)\\n        ti,td=triangles(p)\\n        if not len(ti):continue\\n        dist,near=tree.query(td,k=min(20,len(scene_desc)))\\n        if near.ndim==1:near=near[:,None];dist=dist[:,None]\\n        jobs=[(i,int(j),float(d)) for i,(ns,ds) in enumerate(zip(near,dist)) for j,d in zip(np.atleast_1d(ns),np.atleast_1d(ds))]\\n        # Deterministic class-local selection is invariant to catalog order.\\n        rng=np.random.default_rng(seed+int(hashlib.sha256(name.encode()).hexdigest()[:8],16))\\n        jobs.sort(key=lambda j:j[2])\\n        if len(jobs)>per_class:\\n            first=per_class//2\\n            ids=rng.choice(np.arange(first,len(jobs)),per_class-first,replace=False)\\n            jobs=jobs[:first]+[jobs[i] for i in ids]\\n        best={\\\'name\\\':name,\\\'score\\\':-1e9,\\\'support\\\':0};accepted=0\\n        for i,j,_ in jobs:\\n            src=p[ti[i]];dst=pts[scene_ids[j]]\\n            a=np.c_[src,np.ones(3)]\\n            try:matrix=np.linalg.solve(a,dst)\\n            except np.linalg.LinAlgError:continue\\n            sv=np.linalg.svd(matrix[:2],compute_uv=False)\\n            if sv[-1]<50 or sv[0]>6000 or sv[0]/sv[-1]>8:continue\\n            accepted+=1;mapped=np.c_[p,np.ones(len(p))]@matrix\\n            pairs,res=assignment(mapped,pts,tolerance)\\n            support=len(pairs)\\n            if support<4:continue\\n            # Fitting three nodes alone is never counted as class evidence.\\n            fraction=min(.8,len(pts)*np.pi*tolerance*tolerance/9e6)\\n            surprise=-float(binom.logsf(support-4,max(len(p)-3,1),fraction))/np.log(10)\\n            score=surprise-.5*np.mean(res)/tolerance\\n            if score>best[\\\'score\\\']:\\n                best={\\\'name\\\':name,\\\'score\\\':score,\\\'support\\\':support,\\\'coverage\\\':support/len(p),\\\'mean_residual\\\':float(np.mean(res)),\\\'pairs\\\':pairs,\\\'matrix\\\':matrix.tolist(),\\\'nodes\\\':mapped.tolist()}\\n        best.update(attempted=len(jobs),accepted=accepted,cap_hit=len(jobs)>=per_class)\\n        hypotheses.append(best)\\n    hypotheses.sort(key=lambda h:(-h[\\\'score\\\'],h[\\\'name\\\']))\\n    if not hypotheses or hypotheses[0][\\\'support\\\']<4:\\n        return \\\'unknown\\\',[],{\\\'reason\\\':\\\'no independently verified affine fit\\\',\\\'groups\\\':groups,\\\'hypotheses\\\':hypotheses}\\n    winner=hypotheses[0];supported={j for _,j in winner[\\\'pairs\\\']}\\n    members=[i for i,g in enumerate(groups) if g in supported]\\n    return winner[\\\'name\\\'],members,{\\\'groups\\\':groups,\\\'hypotheses\\\':hypotheses,\\\'score_gap\\\':winner[\\\'score\\\']-hypotheses[1][\\\'score\\\'] if len(hypotheses)>1 else None}\\n\', \'constellation/joint.py\': \'"""Joint localization and recognition.\\n\\nThe frozen recognizer consumed one point per query (the top appearance\\nalternative) and returned only a class plus membership. Measured on the labelled\\nscenes, the correct location is present in the 20 retained alternatives for 90%\\nof present queries but ranks first for only 61% of *figure* queries, with a\\nmedian appearance gap of 0.031. Appearance alone cannot separate them.\\n\\nThis module verifies hypotheses against the whole alternative pool under\\none-to-one constraints per query group, and reports which pool point each query\\nmatched so the caller can adopt the geometrically supported coordinate. A\\nrestricted similarity/reflection branch runs beside the affine branch so a\\nfour-degree-of-freedom explanation is preferred when it fits comparably well.\\n"""\\n\\nimport numpy as np\\nfrom scipy.spatial import cKDTree\\nfrom scipy.stats import binom\\nfrom .geometry import consolidate, triangles\\nfrom .quad import quads\\n\\n# Rejected transform envelope: scene pixels per normalized template unit.\\nMIN_SV, MAX_SV, MAX_ANISO = 50., 6000., 8.\\nSIMILARITY_BONUS = .35   # reward for explaining the data with fewer parameters\\nGEN_LIMIT = 56           # quad generation is O(n^4); cap the seeding point set\\n\\n\\ndef build_pool(alternatives, groups, top_k=8, margin=.15, gap=.03):\\n    """Flatten per-query alternatives into a verification pool tagged by group.\\n\\n    Only *ambiguous* queries contribute more than their top alternative. On the\\n    labelled scenes the rank-0/rank-1 appearance gap separates correctly located\\n    queries (median 0.136) from mislocated ones (median 0.009), so a query whose\\n    top alternative wins clearly is not offered up for geometric relocation.\\n    """\\n    pts, tags, scores, ranks = [], [], [], []\\n    for i, qs in enumerate(alternatives):\\n        if not len(qs):\\n            continue\\n        best = qs[0][2]\\n        ambiguous = len(qs) > 1 and (best - qs[1][2]) < gap\\n        limit = top_k if ambiguous else 1\\n        for r, q in enumerate(qs[:limit]):\\n            if q[2] < best - margin:\\n                break\\n            pts.append(q[:2]); tags.append(groups[i])\\n            scores.append(q[2]); ranks.append(r)\\n    return (np.asarray(pts, dtype=float).reshape(-1, 2), np.asarray(tags, dtype=int),\\n            np.asarray(scores, dtype=float), np.asarray(ranks, dtype=int))\\n\\n\\ndef assignment(transformed, points, tolerance, tags):\\n    """Greedy nearest-first one-to-one match; at most one point per query group."""\\n    if not len(points):\\n        return [], []\\n    d = np.linalg.norm(transformed[:, None, :] - points[None, :, :], axis=2)\\n    candidates = np.argwhere(d < tolerance)\\n    if not len(candidates):\\n        return [], []\\n    order = np.argsort(d[candidates[:, 0], candidates[:, 1]], kind=\\\'stable\\\')\\n    used_node, used_tag, pairs, res = set(), set(), [], []\\n    for k in order:\\n        i, j = candidates[k]\\n        tag = int(tags[j])\\n        if i not in used_node and tag not in used_tag:\\n            used_node.add(i); used_tag.add(tag)\\n            pairs.append((int(i), int(j))); res.append(float(d[i, j]))\\n    return pairs, res\\n\\n\\ndef fit_similarity(src, dst, reflect=False):\\n    """Least-squares similarity (optionally reflected) returned as a 3x2 matrix."""\\n    s = src * np.array([-1., 1.]) if reflect else src\\n    sm, dm = s.mean(0), dst.mean(0)\\n    a, b = s - sm, dst - dm\\n    den = float(np.sum(a * a))\\n    if den < 1e-12:\\n        return None\\n    c = float(np.sum(a * b)) / den\\n    d = float(np.sum(a[:, 0] * b[:, 1] - a[:, 1] * b[:, 0])) / den\\n    r = np.array([[c, d], [-d, c]])\\n    if reflect:\\n        r = np.array([[-1., 0.], [0., 1.]]) @ r\\n    matrix = np.zeros((3, 2))\\n    matrix[:2] = r\\n    matrix[2] = dm - src.mean(0) @ r\\n    return matrix\\n\\n\\ndef fit_affine(src, dst):\\n    try:\\n        return np.linalg.lstsq(np.c_[src, np.ones(len(src))], dst, rcond=None)[0]\\n    except np.linalg.LinAlgError:\\n        return None\\n\\n\\ndef valid(matrix):\\n    if not np.isfinite(matrix).all():\\n        return False\\n    sv = np.linalg.svd(matrix[:2], compute_uv=False)\\n    return not (sv[-1] < MIN_SV or sv[0] > MAX_SV or sv[0] / max(sv[-1], 1e-9) > MAX_ANISO)\\n\\n\\ndef generation_points(alternatives, groups, limit=GEN_LIMIT):\\n    """Seeding point set: highest-scoring rank-0 alternatives, capped for quads."""\\n    items = [(qs[0][2], qs[0][:2]) for qs in alternatives if len(qs)]\\n    if len(items) > limit:\\n        keep = sorted(range(len(items)), key=lambda i: -items[i][0])[:limit]\\n        keep.sort()\\n        items = [items[i] for i in keep]\\n    return np.asarray([xy for _, xy in items], dtype=float).reshape(-1, 2)\\n\\n\\nclass SceneIndex:\\n    """Scene-side invariants and KD-trees, built once and reused for all classes."""\\n\\n    def __init__(self, gen_pts, use_quads=True):\\n        self.tri_ids, tri_desc = triangles(gen_pts)\\n        self.tri_tree = cKDTree(tri_desc) if len(tri_desc) else None\\n        self.quad_ids, self.quad_tree = np.empty((0, 4), int), None\\n        if use_quads and len(gen_pts) >= 4:\\n            qi, qd = quads(gen_pts)\\n            if len(qd):\\n                self.quad_ids, self.quad_tree = qi, cKDTree(qd)\\n\\n    @staticmethod\\n    def _query(tree, ids, desc, k):\\n        if tree is None or not len(desc):\\n            return []\\n        dist, idx = tree.query(desc, k=min(k, tree.n))\\n        dist, idx = np.atleast_2d(dist), np.atleast_2d(idx)\\n        if dist.shape[0] != len(desc):\\n            dist, idx = dist.T, idx.T\\n        out = [(float(d), i, ids[j])\\n               for i in range(len(desc)) for j, d in zip(idx[i], dist[i])]\\n        out.sort(key=lambda x: x[0])\\n        return out\\n\\n    def seeds(self, template, budget, quad_share=.6):\\n        """Quad and triangle seeds are budgeted separately.\\n\\n        Quad invariants are 4-dimensional and triangle invariants 2-dimensional,\\n        so their nearest-neighbour distances are not on a common scale. Merging\\n        and sorting them lets triangles crowd out the more discriminative quads.\\n        """\\n        quad_seeds = []\\n        if self.quad_tree is not None and len(template) >= 4:\\n            ti, td = quads(template)\\n            quad_seeds = [(d, ti[i], sid)\\n                          for d, i, sid in self._query(self.quad_tree, self.quad_ids, td, 4)]\\n        tri_seeds = []\\n        ti3, td3 = triangles(template)\\n        tri_seeds = [(d, ti3[i], sid)\\n                     for d, i, sid in self._query(self.tri_tree, self.tri_ids, td3, 20)]\\n        n_quad = min(len(quad_seeds), int(budget * quad_share))\\n        n_tri = min(len(tri_seeds), budget - n_quad)\\n        n_quad = min(len(quad_seeds), budget - n_tri)\\n        return quad_seeds[:n_quad] + tri_seeds[:n_tri]\\n\\n\\ndef recognize_joint(alternatives, patterns, seed=6643, cap=80000, tolerance=18.,\\n                    top_k=8, margin=.15, shear_penalty=2., auxiliary_map=None,\\n                    models=(\\\'affine\\\',), min_support=4,\\n                    appearance_weight=0., rank_weight=0., gap=.03, diag_top=8,\\n                    score_mode=\\\'binom\\\', sigma=5., size_penalty=0., quad_share=0.,\\n                    aux_weight=3.):\\n    """Return (name, {query index: (x, y)}, diagnostics).\\n\\n    Defaults match what `finalize.finalize_joint` passes in production, so calling\\n    this directly reproduces the shipped configuration. The similarity branch and\\n    quad seeding remain selectable but measured worse; see FINDINGS.md.\\n    """\\n    anchors = np.array([q[0][:2] for q in alternatives if len(q)]).reshape(-1, 2)\\n    if len(anchors) < 3:\\n        return \\\'unknown\\\', {}, {\\\'reason\\\': \\\'fewer than three points\\\', \\\'hypotheses\\\': []}\\n    _, groups = consolidate(anchors)\\n    # groups is indexed over queries that have candidates; map back to query ids.\\n    live = [i for i, q in enumerate(alternatives) if len(q)]\\n    group_of = {q: groups[k] for k, q in enumerate(live)}\\n    pool, tags, pool_scores, pool_ranks = build_pool(alternatives, groups, top_k, margin, gap)\\n    gen_pts = generation_points(alternatives, groups)\\n    if len(gen_pts) < 3 or not len(pool):\\n        return \\\'unknown\\\', {}, {\\\'reason\\\': \\\'insufficient points\\\', \\\'hypotheses\\\': []}\\n\\n    index = SceneIndex(gen_pts)\\n    n_groups = len(set(tags.tolist()))\\n    hypotheses = []\\n    per_class = cap // max(len(patterns), 1)\\n    for name, template in sorted(patterns.items()):\\n        if len(template) < 3:\\n            hypotheses.append({\\\'name\\\': name, \\\'score\\\': -1e9, \\\'support\\\': 0,\\n                               \\\'reason\\\': \\\'pair-only ambiguity\\\'})\\n            continue\\n        # Normalize schematic axis lengths; supplied canvas aspect is arbitrary.\\n        p = (template - template.mean(axis=0)) / np.maximum(np.ptp(template, axis=0), 1e-6)\\n        # Seed selection is deterministic and class-local, so it is invariant to\\n        # catalog order; `seed` is retained only for reporting.\\n        seeds = index.seeds(p, per_class, quad_share)\\n        if not seeds:\\n            hypotheses.append({\\\'name\\\': name, \\\'score\\\': -1e9, \\\'support\\\': 0, \\\'reason\\\': \\\'no seeds\\\'})\\n            continue\\n        best = {\\\'name\\\': name, \\\'score\\\': -1e9, \\\'support\\\': 0}\\n        accepted = 0\\n        for _, ti, si in seeds:\\n            src, dst = p[np.asarray(ti)], gen_pts[np.asarray(si)]\\n            for model in models:\\n                if model == \\\'affine\\\':\\n                    if len(src) == 3:\\n                        try:\\n                            matrix = np.linalg.solve(np.c_[src, np.ones(3)], dst)\\n                        except np.linalg.LinAlgError:\\n                            continue\\n                    else:\\n                        matrix = fit_affine(src, dst)\\n                    bonus = 0.\\n                else:\\n                    matrix = fit_similarity(src, dst)\\n                    mirror = fit_similarity(src, dst, reflect=True)\\n                    if matrix is not None and mirror is not None:\\n                        a = np.c_[src, np.ones(len(src))]\\n                        if np.linalg.norm(a @ mirror - dst) < np.linalg.norm(a @ matrix - dst):\\n                            matrix = mirror\\n                    bonus = SIMILARITY_BONUS\\n                if matrix is None or not valid(matrix):\\n                    continue\\n                accepted += 1\\n                mapped = np.c_[p, np.ones(len(p))] @ matrix\\n                pairs, res = assignment(mapped, pool, max(tolerance, 24.), tags)\\n                if len(pairs) >= min_support:\\n                    for _ in range(3):\\n                        ii, jj = np.array(pairs).T\\n                        refit = (fit_affine(p[ii], pool[jj]) if model == \\\'affine\\\'\\n                                 else fit_similarity(p[ii], pool[jj]))\\n                        if refit is None or not valid(refit):\\n                            break\\n                        matrix = refit\\n                        mapped = np.c_[p, np.ones(len(p))] @ matrix\\n                        pairs, res = assignment(mapped, pool, tolerance, tags)\\n                        if len(pairs) < min_support:\\n                            break\\n                support = len(pairs)\\n                if support < min_support:\\n                    continue\\n                shear = abs(matrix[0] @ matrix[1]) / max(\\n                    np.linalg.norm(matrix[0]) * np.linalg.norm(matrix[1]), 1e-9)\\n                jj = [j for _, j in pairs]\\n                appearance = float(np.mean(pool_scores[jj]))\\n                rank_cost = float(np.mean(pool_ranks[jj]))\\n                if score_mode == \\\'likelihood\\\':\\n                    # Nearly every template can collect four chance matches from a\\n                    # cluttered pool, so support alone does not discriminate. A\\n                    # genuine correspondence lands within ~1px (measured: median\\n                    # 0.78px), whereas a chance match is spread over the whole\\n                    # tolerance disc. Score each matched node by the log ratio of\\n                    # a Gaussian inlier density to the uniform chance density, and\\n                    # charge a fixed cost per template node offered up for matching.\\n                    r = np.asarray(res)\\n                    gain = (np.log(tolerance * tolerance / (2 * sigma * sigma))\\n                            - r * r / (2 * sigma * sigma))\\n                    score = float(gain.sum()) - size_penalty * len(p)\\n                else:\\n                    fraction = min(.8, n_groups * np.pi * tolerance * tolerance / 9e6)\\n                    surprise = -float(binom.logsf(support - min_support,\\n                                                  max(len(p) - 3, 1), fraction)) / np.log(10)\\n                    score = surprise - .5 * np.mean(res) / tolerance\\n                score += (bonus - shear_penalty * shear\\n                          + appearance_weight * appearance - rank_weight * rank_cost)\\n                auxiliary = .5\\n                if auxiliary_map is not None:\\n                    h, w = auxiliary_map.shape\\n                    inside = ((mapped[:, 0] >= 0) & (mapped[:, 1] >= 0)\\n                              & (mapped[:, 0] < w) & (mapped[:, 1] < h))\\n                    supported = {i for i, _ in pairs}\\n                    vals = [float(auxiliary_map[int(pt[1]), int(pt[0])]) if inside[k] else 0.\\n                            for k, pt in enumerate(mapped) if k not in supported]\\n                    auxiliary = float(np.mean(vals)) if vals else .5\\n                    score += aux_weight * (auxiliary - .5)\\n                if score > best[\\\'score\\\']:\\n                    best = {\\\'name\\\': name, \\\'score\\\': float(score), \\\'support\\\': support,\\n                            \\\'model\\\': model, \\\'coverage\\\': support / len(p),\\n                            \\\'mean_residual\\\': float(np.mean(res)), \\\'shear\\\': float(shear),\\n                            \\\'auxiliary\\\': auxiliary, \\\'appearance\\\': appearance,\\n                            \\\'rank_cost\\\': rank_cost, \\\'matrix\\\': matrix.tolist(),\\n                            \\\'nodes\\\': mapped.tolist(),\\n                            \\\'pairs\\\': [(int(i), int(j)) for i, j in pairs]}\\n        best.update(attempted=len(seeds), accepted=accepted, cap_hit=len(seeds) >= per_class)\\n        hypotheses.append(best)\\n\\n    hypotheses.sort(key=lambda h: (-h[\\\'score\\\'], h[\\\'name\\\']))\\n    if not hypotheses or hypotheses[0][\\\'support\\\'] < min_support:\\n        return \\\'unknown\\\', {}, {\\\'reason\\\': \\\'no verified fit\\\',\\n                               \\\'hypotheses\\\': hypotheses[:diag_top], \\\'groups\\\': groups}\\n    winner = hypotheses[0]\\n    by_tag = {int(tags[j]): pool[j] for _, j in winner[\\\'pairs\\\']}\\n    chosen = {i: (float(by_tag[group_of[i]][0]), float(by_tag[group_of[i]][1]))\\n              for i in live if group_of[i] in by_tag}\\n    diag = {\\\'hypotheses\\\': hypotheses[:diag_top], \\\'groups\\\': groups,\\n            \\\'pool_size\\\': int(len(pool)),\\n            \\\'score_gap\\\': (winner[\\\'score\\\'] - hypotheses[1][\\\'score\\\']\\n                          if len(hypotheses) > 1 else None)}\\n    return winner[\\\'name\\\'], chosen, diag\\n\', \'constellation/pipeline.py\': \'"""Classical baselines. Prediction never reads labels or derives classes from scene IDs."""\\nfrom dataclasses import dataclass, asdict\\nfrom pathlib import Path\\nimport time\\nimport cv2\\nimport numpy as np\\nfrom .contracts import ScenePrediction\\n\\nGEOMETRY_MODES = (\\\'radial\\\', \\\'harmonic\\\', \\\'hybrid\\\', \\\'final\\\', \\\'joint\\\')\\nHARMONIC_MODES = (\\\'harmonic\\\', \\\'hybrid\\\', \\\'final\\\', \\\'joint\\\')\\nDENSE_MODES = (\\\'hybrid\\\', \\\'final\\\', \\\'joint\\\')\\nREFINE_MODES = (\\\'final\\\', \\\'joint\\\')\\n\\n\\n@dataclass\\nclass Config:\\n    mode: str = \\\'joint\\\'\\n    threshold: float = .72\\n    alternatives: int = 20\\n    seed: int = 6643\\n    threads: int = 4\\n    # Joint stage: verification tolerance, ambiguity gate, alternatives per\\n    # ambiguous query, hypothesis budget, quad seeding share.\\n    tolerance: float = 18.\\n    gap: float = .03\\n    top_k: int = 8\\n    cap: int = 80000\\n    quad_share: float = 0.\\n    # Verification representation and retained-alternative count. All seven\\n    # labelled present queries with no candidate within 12px reach the proposal\\n    # set and are lost inside verify; its correct proposal ranks 0-1 before\\n    # suppression, so the losses are tail truncation and representation, not\\n    # retrieval. \\\'blur\\\' and 20 reproduce the previous baseline exactly.\\n    verify_rep: str = \\\'blur\\\'\\n    def verify_pair(self, image, patch):\\n        """Preprocessed (scene, patch) pair handed to `retrieval.verify`."""\\n        f = VERIFY_REPS[self.verify_rep]\\n        return f(image.astype(np.float32)), f(patch.astype(np.float32))\\n\\n\\ndef _blur(x, sigma=.6):\\n    return cv2.GaussianBlur(x, (0, 0), sigma)\\n\\n\\ndef _dog(x, lo=.8, hi=2.5):\\n    return cv2.GaussianBlur(x, (0, 0), lo) - cv2.GaussianBlur(x, (0, 0), hi)\\n\\n\\nVERIFY_REPS = {\\\'blur\\\': _blur, \\\'dog\\\': _dog}\\n\\ndef preprocess(image):\\n    x = image.astype(np.float32)\\n    return x-cv2.GaussianBlur(x,(0,0),3)\\n\\ndef distinct_maxima(scores, count=20, radius=16):\\n    scores = scores.copy()\\n    out=[]\\n    for _ in range(count):\\n        _,v,_,(x,y)=cv2.minMaxLoc(scores)\\n        if not np.isfinite(v): break\\n        out.append((x,y,float(v)))\\n        scores[max(0,y-radius):y+radius+1,max(0,x-radius):x+radius+1]=-np.inf\\n    return out\\n\\ndef predict_scene(image, patches, patterns, config):\\n    cv2.setNumThreads(config.threads)\\n    start=time.perf_counter()\\n    sky=preprocess(image)\\n    results=[]; diagnostics=[]\\n    if config.mode in GEOMETRY_MODES:\\n        from .retrieval import build_index,retrieve,verify\\n        if config.mode in HARMONIC_MODES:\\n            from .retrieval import build_harmonic_index as build_index, retrieve_harmonic as retrieve\\n        points,descriptors,stars=build_index(image)\\n    for patch in patches:\\n        q=preprocess(patch)\\n        if config.mode in GEOMETRY_MODES:\\n            proposed=retrieve(patch,points,descriptors)\\n            if config.mode in DENSE_MODES:\\n                from .dense import dense_candidates\\n                proposed=np.unique(np.vstack([proposed,dense_candidates(image,patch)]),axis=0)\\n            scene_rep,patch_rep=config.verify_pair(image,patch)\\n            refined=verify(scene_rep,patch_rep,proposed,config.alternatives)\\n            coarse=refined\\n            if config.mode in REFINE_MODES:\\n                from .refine import refine_candidates\\n                refined=refine_candidates(image,patch,refined)\\n            x,y,s,*_=refined[0]\\n            results.append((x,y,0) if s>=config.threshold else None)\\n            diagnostics.append({"candidates":refined,"appearance_score":s,"coarse_candidates":coarse})\\n            continue\\n        score=cv2.matchTemplate(sky,q,cv2.TM_CCOEFF_NORMED)\\n        candidates=distinct_maxima(score,config.alternatives)\\n        # OpenCV template locations are top-left corners. Pixel-center midpoint\\n        # of an even 32-pixel array is 15.5; the <=12-pixel metric is insensitive\\n        # to the unresolved dataset convention\\\'s possible half-pixel offset.\\n        candidates=[(x+(patch.shape[1]-1)/2,y+(patch.shape[0]-1)/2,s) for x,y,s in candidates]\\n        x,y,s=candidates[0]\\n        results.append((x,y,0) if s>=config.threshold else None)\\n        diagnostics.append({\\\'candidates\\\':candidates,\\\'appearance_score\\\':s})\\n    if config.mode in REFINE_MODES:\\n        from .references import extract_patterns\\n        refined_lists=[q[\\\'candidates\\\'] for q in diagnostics]\\n        coarse_lists=[q[\\\'coarse_candidates\\\'] for q in diagnostics]\\n        references=extract_patterns(patterns)\\n        if config.mode == \\\'joint\\\':\\n            from .finalize import finalize_joint\\n            prediction=finalize_joint(image,refined_lists,coarse_lists,references,\\n                                      config.threshold,config.seed,tolerance=config.tolerance,\\n                                      gap=config.gap,top_k=config.top_k,cap=config.cap,\\n                                      quad_share=config.quad_share)\\n        else:\\n            from .finalize import finalize\\n            prediction=finalize(image,refined_lists,coarse_lists,references,config.threshold,config.seed)\\n        prediction.diagnostics.update(queries=diagnostics,runtime_seconds=time.perf_counter()-start,config=asdict(config),stage=config.mode)\\n        return prediction\\n    from .references import extract_patterns\\n    from .geometry import recognize\\n    present_ids=[i for i,p in enumerate(results) if p is not None]\\n    label,members,geometry=recognize([results[i][:2] for i in present_ids],extract_patterns(patterns),config.seed)\\n    for j in members:\\n        i=present_ids[j];x,y,_=results[i];results[i]=(x,y,1)\\n    return ScenePrediction(results,label,{\\\'queries\\\':diagnostics,\\\'runtime_seconds\\\':time.perf_counter()-start,\\\'config\\\':asdict(config),\\\'stage\\\':config.mode,\\\'geometry\\\':geometry})\\n\', \'constellation/quad.py\': "from itertools import combinations\\nimport numpy as np\\nfrom scipy.spatial import cKDTree\\n\\ndef quads(points):\\n    points=np.asarray(points,dtype=float)\\n    ids=np.array(list(combinations(range(len(points)),4)),dtype=np.int32).reshape(-1,4)\\n    if not len(ids):return ids,np.empty((0,4))\\n    p=points[ids];areas=[]\\n    for i in range(4):\\n        q=np.delete(p,i,axis=1);u=q[:,1]-q[:,0];v=q[:,2]-q[:,0]\\n        areas.append((u[:,0]*v[:,1]-u[:,1]*v[:,0])*(-1)**i)\\n    a=np.stack(areas,axis=1)\\n    largest=np.argmax(abs(a),axis=1);a*=np.where(a[np.arange(len(a)),largest]>=0,1.,-1.)[:,None]\\n    a/=np.maximum(abs(a).sum(axis=1,keepdims=True),1e-12)\\n    order=np.argsort(a,axis=1,kind=\'stable\');a=np.take_along_axis(a,order,axis=1);ids=np.take_along_axis(ids,order,axis=1)\\n    keep=(np.min(abs(a),axis=1)>.015)&(np.min(np.diff(a,axis=1),axis=1)>.002)\\n    return ids[keep],a[keep]\\n\\ndef quad_jobs(template,points,budget):\\n    si,sd=quads(points);ti,td=quads(template)\\n    if not len(sd) or not len(td):return []\\n    distance,index=cKDTree(sd).query(td,k=min(4,len(sd)))\\n    if index.ndim==1:index=index[:,None];distance=distance[:,None]\\n    jobs=[]\\n    for i in range(len(ti)):\\n        for j,d in zip(index[i],distance[i]):jobs.append((float(d),ti[i],si[j]))\\n    jobs.sort(key=lambda x:x[0]);return jobs[:budget]\\n", \'constellation/references.py\': "from pathlib import Path\\nimport cv2\\nimport numpy as np\\n\\ndef extract_patterns(folder):\\n    result={}\\n    for path in sorted(Path(folder).glob(\'*_pattern.png\')):\\n        rgba=cv2.imread(str(path),cv2.IMREAD_UNCHANGED)\\n        # Supplied diagrams encode stars as opaque white disks and edges as green.\\n        mask=((rgba[:,:,:3].min(axis=2)>190)&(rgba[:,:,3]>100)).astype(np.uint8)\\n        n,labels,stats,centers=cv2.connectedComponentsWithStats(mask)\\n        points=centers[1:][stats[1:,cv2.CC_STAT_AREA]>=2]\\n        result[path.stem.removesuffix(\'_pattern\')]=points\\n    return result\\n", \'constellation/refine.py\': \'import cv2\\nimport numpy as np\\n\\ndef refine_candidates(image,patch,candidates):\\n    q=patch.astype(np.float32)\\n    q=cv2.GaussianBlur(q,(0,0),.8)\\n    q=q-cv2.GaussianBlur(q,(0,0),3)\\n    yy,xx=np.mgrid[:32,:32].astype(np.float32)\\n    mask=((xx-15.5)**2+(yy-15.5)**2<14**2).astype(np.uint8)*255\\n    out=[]\\n    for x,y,score,angle,scale in candidates:\\n        crop=cv2.getRectSubPix(image,(64,64),(float(x),float(y))).astype(np.float32)\\n        crop=cv2.GaussianBlur(crop,(0,0),.8)\\n        crop=crop-cv2.GaussianBlur(crop,(0,0),3*scale)\\n        a=np.deg2rad(angle);c=scale*np.cos(a);s=scale*np.sin(a)\\n        warp=np.array([[c,s,31.5-15.5*(c+s)],[-s,c,31.5-15.5*(c-s)]],np.float32)\\n        original=warp.copy()\\n        try:\\n            cc,warp=cv2.findTransformECC(q,crop,warp,cv2.MOTION_AFFINE,(cv2.TERM_CRITERIA_COUNT|cv2.TERM_CRITERIA_EPS,40,1e-4),None,3)\\n            center=warp@np.array([15.5,15.5,1],np.float32)\\n            sv=np.linalg.svd(warp[:,:2],compute_uv=False)\\n            if np.linalg.norm(center-31.5)>5 or min(sv)<.65 or max(sv)>1.6 or max(sv)/min(sv)>1.35:warp=original\\n        except cv2.error:warp=original\\n        sampled=cv2.warpAffine(crop,warp,(32,32),flags=cv2.INTER_LINEAR|cv2.WARP_INVERSE_MAP)\\n        v=q[mask>0];w=sampled[mask>0];v=v-v.mean();w=w-w.mean()\\n        corr=float(v@w/max(np.linalg.norm(v)*np.linalg.norm(w),1e-6))\\n        center=warp@np.array([15.5,15.5,1],np.float32)\\n        out.append((float(x+center[0]-31.5),float(y+center[1]-31.5),corr,float(angle),float(scale)))\\n    return sorted(out,key=lambda p:-p[2])\\n\', \'constellation/retrieval.py\': \'"""Rotation-invariant radial proposals followed by masked rotation/scale verification."""\\nimport cv2\\nimport numpy as np\\nfrom scipy.spatial import cKDTree\\n\\ndef normalize(x):\\n    x=x-x.mean(axis=-1,keepdims=True)\\n    return x/np.maximum(np.linalg.norm(x,axis=-1,keepdims=True),1e-6)\\n\\ndef radial_maps(image):\\n    yy,xx=np.mgrid[-20:21,-20:21];r=np.hypot(xx,yy)\\n    maps=[]\\n    for radius in np.arange(0,19,2):\\n        kernel=np.exp(-.5*((r-radius)/1.15)**2).astype(np.float32);kernel/=kernel.sum()\\n        maps.append(cv2.filter2D(image,-1,kernel))\\n    return np.stack(maps,axis=-1)\\n\\ndef build_index(image,stride=4):\\n    raw=image.astype(np.float32)\\n    maps=radial_maps(raw)\\n    yy,xx=np.mgrid[20:raw.shape[0]-20:stride,20:raw.shape[1]-20:stride]\\n    dense=np.c_[xx.ravel(),yy.ravel()]\\n    blur=cv2.GaussianBlur(raw,(0,0),.8)\\n    dog=blur-cv2.GaussianBlur(raw,(0,0),2.5)\\n    maxima=(dog==cv2.dilate(dog,np.ones((5,5),np.uint8)))&(dog>1.5)\\n    y,x=np.where(maxima);keep=(x>=20)&(x<raw.shape[1]-20)&(y>=20)&(y<raw.shape[0]-20)\\n    stars=np.c_[x[keep],y[keep]]\\n    points=np.unique(np.vstack([dense,stars]),axis=0)\\n    desc=normalize(maps[points[:,1],points[:,0]])\\n    return points,desc,stars\\n\\ndef query_descriptors(patch):\\n    theta=np.arange(64)*2*np.pi/64\\n    out=[]\\n    for scale in (.75,.87,1.,1.15,1.33):\\n        radius=np.arange(0,19,2)/scale\\n        mx=(15.5+radius[:,None]*np.cos(theta)).astype(np.float32)\\n        my=(15.5+radius[:,None]*np.sin(theta)).astype(np.float32)\\n        values=cv2.remap(patch.astype(np.float32),mx,my,cv2.INTER_LINEAR,borderMode=cv2.BORDER_REFLECT_101)\\n        out.append(values.mean(axis=1))\\n    return normalize(np.array(out))\\n\\ndef retrieve(patch,points,descriptors,budget=1000):\\n    d=query_descriptors(patch)\\n    scores=np.max(descriptors@d.T,axis=1)\\n    ids=np.argpartition(scores,-budget)[-budget:]\\n    return points[ids[np.argsort(scores[ids])[::-1]]]\\n\\ndef verify(image,patch,candidates,keep=20):\\n    # Sample each sky neighborhood and compare to transformed query over a\\n    # common circular support; float32 maps avoid OpenCV sampling dtype errors.\\n    yy,xx=np.mgrid[-12:13:2,-12:13:2].astype(np.float32)\\n    mask=(xx*xx+yy*yy<=144);xx=xx[mask];yy=yy[mask]\\n    xmap=candidates[:,0,None].astype(np.float32)+xx\\n    ymap=candidates[:,1,None].astype(np.float32)+yy\\n    samples=cv2.remap(image,xmap,ymap,cv2.INTER_LINEAR)\\n    samples=normalize(samples)\\n    templates=[];transforms=[]\\n    for scale in (.75,.87,1.,1.15,1.33):\\n        for angle in np.arange(0,360,15):\\n            a=np.deg2rad(angle)\\n            mx=(15.5+(np.cos(a)*xx-np.sin(a)*yy)/scale).astype(np.float32)[None,:]\\n            my=(15.5+(np.sin(a)*xx+np.cos(a)*yy)/scale).astype(np.float32)[None,:]\\n            valid=(mx>=0)&(mx<=31)&(my>=0)&(my<=31)\\n            if not valid.all():continue\\n            v=cv2.remap(patch,mx,my,cv2.INTER_LINEAR).ravel()\\n            templates.append(v);transforms.append((float(angle),float(scale)))\\n    templates=normalize(np.array(templates))\\n    scores=samples@templates.T\\n    best=scores.max(axis=1); order=np.argsort(best)[::-1]\\n    selected=[]\\n    for idx in order:\\n        pt=candidates[idx]\\n        if any(np.linalg.norm(pt-np.array(s[:2]))<8 for s in selected):continue\\n        angle,scale=transforms[int(scores[idx].argmax())]\\n        selected.append((float(pt[0]),float(pt[1]),float(best[idx]),angle,scale))\\n        if len(selected)>=keep:break\\n    # Refine each alternative around its immutable coarse pose.\\n    refined=[]\\n    for x,y,_,angle,scale in selected:\\n        best=(-2,None)\\n        for da in (-7.5,0,7.5):\\n            a=np.deg2rad(angle+da)\\n            for ds in (.94,1,1.06):\\n                mx=(15.5+(np.cos(a)*xx-np.sin(a)*yy)/(scale*ds)).astype(np.float32)[None,:]\\n                my=(15.5+(np.sin(a)*xx+np.cos(a)*yy)/(scale*ds)).astype(np.float32)[None,:]\\n                if mx.min()<0 or mx.max()>31 or my.min()<0 or my.max()>31:continue\\n                q=normalize(cv2.remap(patch,mx,my,cv2.INTER_LINEAR))\\n                offsets=np.array([(dx,dy) for dx in (-2,-1,0,1,2) for dy in (-2,-1,0,1,2)],np.float32)\\n                sm=cv2.remap(image,(x+offsets[:,0,None]+xx).astype(np.float32),(y+offsets[:,1,None]+yy).astype(np.float32),cv2.INTER_LINEAR)\\n                corr=normalize(sm)@q.ravel();idx=int(corr.argmax())\\n                if corr[idx]>best[0]:best=(float(corr[idx]),(x+float(offsets[idx,0]),y+float(offsets[idx,1]),float(corr[idx]),angle+da,scale*ds))\\n        if best[1] is not None:refined.append(best[1])\\n    return sorted(refined,key=lambda x:-x[2])\\n\\ndef harmonic_features(image,points,scale=1.):\\n    """Circular harmonic magnitudes, invariant to in-plane rotation."""\\n    size=int(np.ceil(17*scale)); yy,xx=np.mgrid[-size:size+1,-size:size+1]\\n    radius=np.hypot(xx,yy)/scale;theta=np.arctan2(yy,xx)\\n    features=[]\\n    for r in (2,5,8,11,14):\\n        ring=np.exp(-.5*((radius-r)/1.2)**2);ring/=ring.sum()\\n        for order in (0,1,2,3):\\n            real=cv2.filter2D(image,-1,(ring*np.cos(order*theta)).astype(np.float32))\\n            rv=real[points[:,1],points[:,0]]\\n            if order:\\n                imag=cv2.filter2D(image,-1,(ring*np.sin(order*theta)).astype(np.float32))\\n                iv=imag[points[:,1],points[:,0]]\\n                features.append(np.hypot(rv,iv))\\n            else:features.append(rv)\\n    feat=np.stack(features,axis=-1)\\n    feat[:,::4]-=feat[:,::4].mean(axis=1,keepdims=True)\\n    return feat/np.maximum(np.linalg.norm(feat,axis=1,keepdims=True),1e-6)\\n\\ndef build_harmonic_index(image,stride=4):\\n    raw=cv2.GaussianBlur(image.astype(np.float32),(0,0),.6)\\n    yy,xx=np.mgrid[20:raw.shape[0]-20:stride,20:raw.shape[1]-20:stride]\\n    dense=np.c_[xx.ravel(),yy.ravel()]\\n    dog=raw-cv2.GaussianBlur(raw,(0,0),2.5)\\n    maxima=(dog==cv2.dilate(dog,np.ones((5,5),np.uint8)))&(dog>1.5)\\n    y,x=np.where(maxima);keep=(x>=20)&(x<raw.shape[1]-20)&(y>=20)&(y<raw.shape[0]-20)\\n    stars=np.c_[x[keep],y[keep]]\\n    points=np.unique(np.vstack([dense,stars]),axis=0)\\n    return points,harmonic_features(raw,points),stars\\n\\ndef retrieve_harmonic(patch,points,descriptors,budget=2000):\\n    raw=cv2.GaussianBlur(patch.astype(np.float32),(0,0),.6)\\n    ds=np.concatenate([harmonic_features(raw,np.array([[15,15],[16,16],[15,16],[16,15]]),scale) for scale in (.75,.87,1.,1.15,1.33)])\\n    scores=np.max(descriptors@ds.T,axis=1)\\n    ids=np.argpartition(scores,-budget)[-budget:]\\n    return points[ids[np.argsort(scores[ids])[::-1]]]\\n\', \'run.py\': \'#!/usr/bin/env python3\\n"""Reproducible classical inference, evaluation and submission entrypoint."""\\nimport argparse\\nfrom concurrent.futures import ProcessPoolExecutor,as_completed\\nfrom dataclasses import asdict\\nimport csv,hashlib,json,platform,resource\\nfrom pathlib import Path\\nimport cv2\\nimport numpy as np\\nfrom constellation.contracts import read_truth,evaluate,write_submission\\nfrom constellation.pipeline import Config,predict_scene\\n\\ndef process_row(root,split,row,config,smoke,output):\\n    scene=Path(root)/split/row[\\\'Id\\\']\\n    images=list(scene.glob(\\\'*_image.png\\\'))\\n    if len(images)!=1:raise ValueError(f\\\'Expected one sky in {scene}\\\')\\n    image=cv2.imread(str(images[0]),0)\\n    n=int(row[\\\'n_patches\\\']);n=min(n,2) if smoke else n\\n    patches=[cv2.imread(str(scene/\\\'patches\\\'/f\\\'patch_{i:02}.png\\\'),0) for i in range(1,n+1)]\\n    if image is None or any(q is None for q in patches):raise ValueError(\\\'Missing images\\\')\\n    if image.shape!=(3000,3000) or any(q.shape!=(32,32) for q in patches):raise ValueError(\\\'Unexpected dimensions\\\')\\n    prediction=predict_scene(image,patches,Path(root)/\\\'patterns\\\',config)\\n    prediction.diagnostics[\\\'peak_rss_platform_units\\\']=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss\\n    (Path(output)/f"{row[\\\'Id\\\']}.json").write_text(json.dumps(asdict(prediction),indent=2))\\n    print(row[\\\'Id\\\'],round(prediction.diagnostics[\\\'runtime_seconds\\\'],2),\\\'seconds\\\',flush=True)\\n    return row[\\\'Id\\\'],prediction\\n\\ndef source_hash():\\n    here=Path(__file__).resolve().parent\\n    source=b\\\'\\\'.join(f.read_bytes() for f in sorted((here/\\\'constellation\\\').glob(\\\'*.py\\\')))+Path(__file__).read_bytes()\\n    return hashlib.sha256(source).hexdigest()\\n\\ndef main():\\n    p=argparse.ArgumentParser()\\n    p.add_argument(\\\'--data\\\',type=Path,default=Path(\\\'.\\\'))\\n    p.add_argument(\\\'--mode\\\',choices=[\\\'smoke\\\',\\\'evaluate\\\',\\\'submission\\\'],default=\\\'evaluate\\\')\\n    p.add_argument(\\\'--output\\\',type=Path,default=Path(\\\'outputs/final\\\'))\\n    p.add_argument(\\\'--pipeline\\\',choices=[\\\'a0\\\',\\\'radial\\\',\\\'harmonic\\\',\\\'hybrid\\\',\\\'final\\\',\\\'joint\\\'],default=\\\'joint\\\')\\n    p.add_argument(\\\'--threshold\\\',type=float,default=.72)\\n    p.add_argument(\\\'--workers\\\',type=int,default=1)\\n    p.add_argument(\\\'--threads\\\',type=int,default=2)\\n    p.add_argument(\\\'--verify-rep\\\',choices=[\\\'blur\\\',\\\'dog\\\'],default=\\\'blur\\\')\\n    p.add_argument(\\\'--alternatives\\\',type=int,default=20)\\n    args=p.parse_args()\\n    if not 0<=args.threshold<=1 or args.workers<1:raise ValueError(\\\'Invalid configuration\\\')\\n    root=args.data.resolve()\\n    if not (root/\\\'patterns\\\').exists() and (root/\\\'participant\\\').exists():root=root/\\\'participant\\\'\\n    args.output.mkdir(parents=True,exist_ok=True)\\n    config=Config(threshold=args.threshold,mode=args.pipeline,threads=args.threads,verify_rep=args.verify_rep,alternatives=args.alternatives)\\n    template=root/(\\\'sample_submission.csv\\\' if args.mode==\\\'submission\\\' else \\\'train_ground_truth.csv\\\')\\n    with template.open() as f:rows=list(csv.DictReader(f))\\n    if len({r[\\\'Id\\\'] for r in rows})!=len(rows):raise ValueError(\\\'Duplicate IDs\\\')\\n    if args.mode==\\\'smoke\\\':rows=rows[:1]\\n    digest=source_hash()\\n    manifest={\\\'config\\\':asdict(config),\\\'python\\\':platform.python_version(),\\\'numpy\\\':np.__version__,\\\'opencv\\\':cv2.__version__,\\\'source_sha256\\\':digest,\\\'mode\\\':args.mode,\\\'workers\\\':args.workers,\\\'status\\\':\\\'running\\\'}\\n    (args.output/\\\'manifest.json\\\').write_text(json.dumps(manifest,indent=2))\\n    split=\\\'validation\\\' if args.mode==\\\'submission\\\' else \\\'train\\\'\\n    predictions={}\\n    if args.workers==1:\\n        for row in rows:\\n            name,pred=process_row(root,split,row,config,args.mode==\\\'smoke\\\',args.output)\\n            predictions[name]=pred\\n    else:\\n        with ProcessPoolExecutor(max_workers=args.workers) as pool:\\n            futures=[pool.submit(process_row,root,split,row,config,args.mode==\\\'smoke\\\',args.output) for row in rows]\\n            for future in as_completed(futures):\\n                name,pred=future.result();predictions[name]=pred\\n    if source_hash()!=digest:raise RuntimeError(\\\'Source changed during inference; rerun from frozen source\\\')\\n    if args.mode==\\\'evaluate\\\':\\n        metrics=evaluate(predictions,read_truth(template))\\n        (args.output/\\\'metrics.json\\\').write_text(json.dumps(metrics,indent=2));print(json.dumps(metrics,indent=2))\\n    if args.mode!=\\\'smoke\\\':write_submission(predictions,template,args.output/\\\'submission.csv\\\')\\n    manifest.update(status=\\\'complete\\\',peak_rss_platform_units=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss)\\n    (args.output/\\\'manifest.json\\\').write_text(json.dumps(manifest,indent=2))\\nif __name__==\\\'__main__\\\':main()\\n\', \'audit.py\': "from pathlib import Path\\nimport csv,hashlib,json\\nimport cv2\\nimport numpy as np\\nfrom PIL import Image,ImageDraw\\nfrom constellation.contracts import read_truth\\nout=Path(\'outputs/audit\');out.mkdir(parents=True,exist_ok=True)\\nrows=[]; hashes={}; duplicates=[]\\nfor path in sorted(Path(\'.\').glob(\'**/*.png\')):\\n    if path.parts[0] not in (\'train\',\'validation\',\'patterns\'):continue\\n    a=np.array(Image.open(path));h=hashlib.sha256(a.tobytes()).hexdigest()\\n    if h in hashes:duplicates.append([str(path),hashes[h]])\\n    hashes[h]=str(path)\\n    rows.append(dict(path=str(path),shape=str(a.shape),dtype=str(a.dtype),sha256=h))\\nwith (out/\'inventory.csv\').open(\'w\') as f:\\n    w=csv.DictWriter(f,fieldnames=rows[0]);w.writeheader();w.writerows(rows)\\ntruth=read_truth(\'train_ground_truth.csv\')\\nsummary={n:dict(total=len(p.patches),figure=sum(x is not None and x[2]==1 for x in p.patches),off_figure=sum(x is not None and x[2]==0 for x in p.patches),absent=sum(x is None for x in p.patches)) for n,p in truth.items()}\\n(out/\'summary.json\').write_text(json.dumps(dict(files=len(rows),duplicates=duplicates,labels=summary),indent=2))\\npaths=sorted(Path(\'patterns\').glob(\'*.png\'))\\ncanvas=Image.new(\'RGB\',(1200,1600),\'#202030\');draw=ImageDraw.Draw(canvas)\\nfor i,path in enumerate(paths):\\n    im=Image.open(path).convert(\'RGBA\');bg=Image.new(\'RGBA\',im.size,\'white\');bg.alpha_composite(im);bg=bg.convert(\'RGB\');bg.thumbnail((185,170))\\n    x=(i%6)*200;y=(i//6)*200;canvas.paste(bg,(x,y+20));draw.text((x+2,y+2),path.stem.replace(\'_pattern\',\'\'),fill=\'white\')\\ncanvas.save(out/\'patterns.jpg\')\\nprint(json.dumps(summary,indent=2));print(\'Files\',len(rows),\'duplicates\',len(duplicates))\\n", \'calibrate.py\': \'"""Leave-one-scene-out threshold calibration using saved blind candidates."""\\nimport argparse,json\\nfrom pathlib import Path\\nimport numpy as np\\nfrom constellation.contracts import ScenePrediction,evaluate,read_truth\\nfrom constellation.geometry import recognize\\nfrom constellation.references import extract_patterns\\n\\ndef prediction(saved,threshold,patterns=None):\\n    patches=[]\\n    for q in saved[\\\'diagnostics\\\'][\\\'queries\\\']:\\n        x,y,score,*_=q[\\\'candidates\\\'][0]\\n        patches.append((x,y,0) if score>=threshold else None)\\n    label=\\\'unknown\\\';d={}\\n    if patterns is not None:\\n        ids=[i for i,p in enumerate(patches) if p is not None]\\n        label,m,d=recognize([patches[i][:2] for i in ids],patterns)\\n        for j in m:\\n            i=ids[j];x,y,_=patches[i];patches[i]=(x,y,1)\\n    return ScenePrediction(patches,label,d)\\n\\ndef main():\\n    parser=argparse.ArgumentParser();parser.add_argument(\\\'--input\\\',type=Path,required=True);args=parser.parse_args()\\n    truth=read_truth(\\\'train_ground_truth.csv\\\');saved={n:json.loads((args.input/f\\\'{n}.json\\\').read_text()) for n in truth}\\n    thresholds=np.arange(.65,.991,.01);scores={};all_pred={}\\n    # Threshold fitting deliberately excludes identification to avoid rerunning\\n    # expensive geometry and to keep presence calibration appearance-driven.\\n    for threshold in thresholds:\\n        pred={n:prediction(s,float(threshold)) for n,s in saved.items()}\\n        scores[float(threshold)]=evaluate(pred,truth)\\n    folds={};held={};patterns=extract_patterns(\\\'patterns\\\')\\n    for name in truth:\\n        dev=[n for n in truth if n!=name]\\n        threshold=max(scores,key=lambda t:np.mean([scores[t][\\\'scenes\\\'][n][\\\'score\\\'] for n in dev]))\\n        held[name]=prediction(saved[name],threshold,patterns)\\n        folds[name]={\\\'development_scenes\\\':dev,\\\'threshold\\\':threshold}\\n    chosen=max(scores,key=lambda t:scores[t][\\\'mean\\\'][\\\'score\\\'])\\n    result={\\\'folds\\\':folds,\\\'held_out\\\':evaluate(held,truth),\\\'final_threshold\\\':chosen,\\\'selection\\\':\\\'maximize development-scene mean of 0.25 presence + 0.20 localization + 0.25 recovery; geometry evaluated after threshold selection\\\',\\\'limitation\\\':\\\'All three scenes informed iterative method development; LOSO threshold results are not an untouched estimate of method selection generalization.\\\'}\\n    (args.input/\\\'calibration.json\\\').write_text(json.dumps(result,indent=2));print(json.dumps(result,indent=2))\\nif __name__==\\\'__main__\\\':main()\\n\', \'geometry_oracle.py\': "import json\\nfrom pathlib import Path\\nfrom constellation.contracts import read_truth\\nfrom constellation.references import extract_patterns\\nfrom constellation.geometry import recognize\\ntruth=read_truth(\'train_ground_truth.csv\');patterns=extract_patterns(\'patterns\');out={}\\nprint({n:len(p) for n,p in patterns.items()},flush=True)\\nfor n,t in truth.items():\\n    out[n]={}\\n    for mode in (\'present\',\'figure\'):\\n        points=[p[:2] for p in t.patches if p is not None and (mode==\'present\' or p[2]==1)]\\n        label,m,d=recognize(points,patterns)\\n        out[n][mode]={\'prediction\':label,\'diagnostics\':d}\\n        print(n,mode,label,[(h[\'name\'],h[\'support\'],round(h[\'score\'],2)) for h in d[\'hypotheses\'][:3]],flush=True)\\nPath(\'outputs/geometry_oracles.json\').write_text(json.dumps(out,indent=2))\\n", \'requirements.txt\': \'numpy>=2.0\\nscipy>=1.14\\nopencv-python-headless>=4.10\\nPillow>=10\\nnbformat>=5.10\\nkaggle>=1.6\\n\'}\nwith tempfile.TemporaryDirectory(prefix=\'constellation-\') as directory:\n    root=pathlib.Path(directory)\n    for name,source in SOURCES.items():\n        path=root/name;path.parent.mkdir(parents=True,exist_ok=True);path.write_text(source)\n    sys.path.insert(0,str(root))\n    runpy.run_path(str(root/\'run.py\'),run_name=\'__main__\')\n'
export=OUTPUT/'constellation_inference.py';export.write_text(STANDALONE)
print('Standalone script:',export)
if (OUTPUT/'submission.csv').exists():print('CSV:',OUTPUT/'submission.csv')